In [ ]:
# ============================================================
# QUOTATION EXTRACTION PIPELINE v3.0.0
# Production-Ready | Clean Architecture | Fully Tested
# ============================================================

# CELL 1: Core Configuration & Imports
import os
import sys
import json
import re
import time
import hashlib
import zipfile
import sqlite3
import gc
import traceback
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any, Set
from dataclasses import dataclass, field, asdict
from collections import defaultdict
from enum import Enum
from difflib import SequenceMatcher
from typing import Optional

# Install dependencies
!pip install python-docx pdfplumber pillow pytesseract openpyxl pandas -q

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Configuration
@dataclass
class Config:
    input_zip: str = '/content/drive/MyDrive/YOUR_FOLDER_NAMEn_SIMPL.zip'
    output_dir: str = '/content/drive/MyDrive/YOUR_FOLDER_NAMEn_Output_v3'
    batch_size: int = 10
    file_delay: float = 0.5

    # Validation thresholds
    min_total_amount: float = 1000
    max_total_amount: float = 50_000_000

    # Confidence thresholds
    auto_review_threshold: float = 0.60
    high_confidence_threshold: float = 0.85

    # Version
    version: str = "3.0.0"

config = Config()
OUTPUT_DIR = Path(config.output_dir)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"""
╔══════════════════════════════════════════════════════════════╗
║     QUOTATION EXTRACTION PIPELINE v{config.version}                    ║
║     Clean Architecture | Production Ready                      ║
╚══════════════════════════════════════════════════════════════╝
""")

In [ ]:
# CELL 2: Data Models
@dataclass
class ExtractedField:
    """Single extracted field with metadata"""
    value: Any
    confidence: float = 0.0
    source: str = ""  # regex, llm, filename, table
    raw_value: str = ""

    def to_dict(self) -> Dict:
        return {
            'value': self.value,
            'confidence': self.confidence,
            'source': self.source
        }

@dataclass
class QuotationRecord:
    """Complete quotation extraction record"""
    # Core fields
    quotation_no: Optional[ExtractedField] = None
    date: Optional[ExtractedField] = None
    client: Optional[ExtractedField] = None
    city: Optional[ExtractedField] = None
    product_type: Optional[ExtractedField] = None
    total_amount: Optional[ExtractedField] = None
    gst: Optional[ExtractedField] = None
    subtotal: Optional[ExtractedField] = None
    salesperson: Optional[ExtractedField] = None
    area_sqft: Optional[ExtractedField] = None

    # Metadata
    filename: str = ""
    extraction_method: str = ""  # direct, ocr
    overall_confidence: float = 0.0
    needs_review: bool = False
    review_reasons: List[str] = field(default_factory=list)
    extracted_at: str = ""

    def get_value(self, field: str) -> Any:
        attr = getattr(self, field, None)
        return attr.value if attr else None

    def calculate_confidence(self) -> float:
        """Calculate overall confidence from critical fields"""
        weights = {
            'quotation_no': 0.20,
            'client': 0.25,
            'total_amount': 0.30,
            'date': 0.15,
            'product_type': 0.10
        }

        total_weight = 0
        weighted_sum = 0

        for field, weight in weights.items():
            attr = getattr(self, field, None)
            if attr and attr.confidence > 0:
                weighted_sum += attr.confidence * weight
                total_weight += weight

        self.overall_confidence = weighted_sum / total_weight if total_weight > 0 else 0.0
        self.needs_review = self.overall_confidence < config.auto_review_threshold

        return self.overall_confidence

    def to_dict(self) -> Dict:
        result = {
            'filename': self.filename,
            'overall_confidence': self.overall_confidence,
            'needs_review': self.needs_review,
            'review_reasons': self.review_reasons,
            'extracted_at': self.extracted_at
        }

        for field in ['quotation_no', 'date', 'client', 'city', 'product_type',
                      'total_amount', 'gst', 'subtotal', 'salesperson', 'area_sqft']:
            attr = getattr(self, field)
            result[field] = attr.to_dict() if attr else {'value': None, 'confidence': 0}

        return result

print("✅ Data models defined")

In [ ]:
# CELL 3: Master Data Dictionaries (Fixed)
class MasterData:
    """Client, product, and city master data with normalization"""

    def __init__(self):
        self._init_clients()
        self._init_products()
        self._init_cities()
        self._init_salespersons()

    def _init_clients(self):
        # Full customer list from Correctreport (all 165 real names + key variant aliases)
        self.clients = {
            'Client_029':              {'aliases': ['Unit Confiance', 'Confiance', 'The Unit'], 'confidence': 1.0},
            'Client_121':                     {'aliases': ['Nath Awnings', 'Nath Awning Pvt Ltd'], 'confidence': 1.0},
            'Client_014':           {'aliases': ['Megavent Technologies Pvt Ltd', 'Megavent Tech'], 'confidence': 1.0},
            'Client_030':              {'aliases': ['Megavent Lifespaces', 'Megavent Lifespaces Pvt Ltd'], 'confidence': 1.0},
            'Client_017':            {'aliases': ['Whiteshade Solution', 'M/s. White Shade', 'White Shade'], 'confidence': 1.0},
            'Client_083':                  {'aliases': ['Shreya Tensile.'], 'confidence': 1.0},
            'Client_018':            {'aliases': ['Mr. Rakesh Roshan Ji .. 1', 'Mr. H Roshan'], 'confidence': 1.0},
            'Client_044':               {'aliases': [], 'confidence': 1.0},
            'Client_023':             {'aliases': ['Naik and Associates.'], 'confidence': 1.0},
            'Client_054':                {'aliases': ['Mr. Rajesh Sutar.', 'Mr. Rajesh Suthar'], 'confidence': 1.0},
            'Client_019':            {'aliases': ['Sai Developers'], 'confidence': 1.0},
            'Client_084':                  {'aliases': [], 'confidence': 1.0},
            'Client_136':                       {'aliases': [], 'confidence': 1.0},
            'Client_097':                   {'aliases': ['Taj Lands End.'], 'confidence': 1.0},
            'Client_098':                   {'aliases': [], 'confidence': 1.0},
            'Client_007':         {'aliases': [], 'confidence': 1.0},
            'Client_055':                {'aliases': [], 'confidence': 1.0},
            'Client_045':               {'aliases': [], 'confidence': 1.0},
            'Client_031':              {'aliases': ['Am Infra'], 'confidence': 1.0},
            'Client_032':              {'aliases': [], 'confidence': 1.0},
            'Client_015':           {'aliases': [], 'confidence': 1.0},
            'Client_150':                          {'aliases': [], 'confidence': 1.0},
            'Client_085':                  {'aliases': [], 'confidence': 1.0},
            'Client_046':               {'aliases': [], 'confidence': 1.0},
            'Client_108':                    {'aliases': [], 'confidence': 1.0},
            'Client_122':                     {'aliases': [], 'confidence': 1.0},
            'Client_056':                {'aliases': [], 'confidence': 1.0},
            'Client_047':               {'aliases': ['Ar. Maya Savarker (1)'], 'confidence': 1.0},
            'Client_057':                {'aliases': [], 'confidence': 1.0},
            'Client_033':              {'aliases': ['Ar. Prassanna Hegde'], 'confidence': 1.0},
            'Client_058':                {'aliases': [], 'confidence': 1.0},
            'Client_086':                  {'aliases': [], 'confidence': 1.0},
            'Client_068':                 {'aliases': [], 'confidence': 1.0},
            'Client_123':                     {'aliases': [], 'confidence': 1.0},
            'Client_124':                     {'aliases': [], 'confidence': 1.0},
            'Client_069':                 {'aliases': [], 'confidence': 1.0},
            'Client_034':              {'aliases': ['Balaji Fabrication Surat'], 'confidence': 1.0},
            'Client_145':                         {'aliases': [], 'confidence': 1.0},
            'Client_109':                    {'aliases': [], 'confidence': 1.0},
            'Client_059':                {'aliases': [], 'confidence': 1.0},
            'Client_035':              {'aliases': [], 'confidence': 1.0},
            'Client_110':                    {'aliases': [], 'confidence': 1.0},
            'Client_020':            {'aliases': [], 'confidence': 1.0},
            'Client_010':          {'aliases': [], 'confidence': 1.0},
            'Client_036':              {'aliases': [], 'confidence': 1.0},
            'Client_151':                          {'aliases': [], 'confidence': 1.0},
            'Client_087':                  {'aliases': [], 'confidence': 1.0},
            'Client_141':                        {'aliases': [], 'confidence': 1.0},
            'Client_146':                         {'aliases': ['G Décor'], 'confidence': 1.0},
            'Client_048':               {'aliases': ['Global Insulation & Marine Suppliers'], 'confidence': 1.0},
            'Client_088':                  {'aliases': [], 'confidence': 1.0},
            'Client_049':               {'aliases': [], 'confidence': 1.0},
            'Client_147':                         {'aliases': [], 'confidence': 1.0},
            'Client_129':                      {'aliases': [], 'confidence': 1.0},
            'Hpcl':                            {'aliases': [], 'confidence': 1.0},
            'Client_111':                    {'aliases': ['Hr Residency 06-12-2025', 'Hr Residency 06-12-2025[1]'], 'confidence': 1.0},
            'Client_089':                  {'aliases': [], 'confidence': 1.0},
            'Client_050':               {'aliases': ['Innovative Lounge.'], 'confidence': 1.0},
            'Client_112':                    {'aliases': [], 'confidence': 1.0},
            'Client_113':                    {'aliases': [' K S Interior'], 'confidence': 1.0},
            'Client_125':                     {'aliases': [], 'confidence': 1.0},
            'Client_024':             {'aliases': ['Karan Collaborative.'], 'confidence': 1.0},
            'Client_099':                   {'aliases': ['Kings Canopies'], 'confidence': 1.0},
            'Client_070':                 {'aliases': [], 'confidence': 1.0},
            'Client_025':             {'aliases': [], 'confidence': 1.0},
            'Client_100':                   {'aliases': [], 'confidence': 1.0},
            'Client_026':             {'aliases': ['Mr. Adnan Poonawala.', 'Mr. Adernianwala'], 'confidence': 1.0},
            'Client_060':                {'aliases': [], 'confidence': 1.0},
            'Client_071':                 {'aliases': [], 'confidence': 1.0},
            'Client_142':                        {'aliases': [], 'confidence': 1.0},
            'Client_072':                 {'aliases': [], 'confidence': 1.0},
            'Client_061':                {'aliases': [], 'confidence': 1.0},
            'Client_073':                 {'aliases': [], 'confidence': 1.0},
            'Client_130':                      {'aliases': [], 'confidence': 1.0},
            'Client_051':               {'aliases': [], 'confidence': 1.0},
            'Client_027':             {'aliases': [], 'confidence': 1.0},
            'Client_090':                  {'aliases': [], 'confidence': 1.0},
            'Client_091':                  {'aliases': [], 'confidence': 1.0},
            'Client_062':                {'aliases': [], 'confidence': 1.0},
            'Client_021':            {'aliases': [], 'confidence': 1.0},
            'Client_131':                      {'aliases': [], 'confidence': 1.0},
            'Client_074':                 {'aliases': [], 'confidence': 1.0},
            'Client_126':                     {'aliases': [], 'confidence': 1.0},
            'Client_037':              {'aliases': ['Mr. Paresh Mahdev'], 'confidence': 1.0},
            'Client_132':                      {'aliases': [], 'confidence': 1.0},
            'Client_114':                    {'aliases': [], 'confidence': 1.0},
            'Client_137':                       {'aliases': [], 'confidence': 1.0},
            'Client_063':                {'aliases': [], 'confidence': 1.0},
            'Client_092':                  {'aliases': [], 'confidence': 1.0},
            'Client_028':             {'aliases': ['Rishabh Baldota'], 'confidence': 1.0},
            'Client_011':          {'aliases': ['Sujit V', 'Sujit Vishwakarma'], 'confidence': 1.0},
            'Client_075':                 {'aliases': [], 'confidence': 1.0},
            'Client_133':                      {'aliases': [], 'confidence': 1.0},
            'Client_138':                       {'aliases': [], 'confidence': 1.0},
            'Client_038':              {'aliases': [], 'confidence': 1.0},
            'Client_052':               {'aliases': [], 'confidence': 1.0},
            'Client_101':                   {'aliases': [], 'confidence': 1.0},
            'Client_076':                 {'aliases': [], 'confidence': 1.0},
            'Client_012':          {'aliases': ['Systematic Systems'], 'confidence': 1.0},
            'Client_053':               {'aliases': [], 'confidence': 1.0},
            'Client_115':                    {'aliases': [], 'confidence': 1.0},
            'Client_127':                     {'aliases': [], 'confidence': 1.0},
            'Client_116':                    {'aliases': [], 'confidence': 1.0},
            'Client_064':                {'aliases': [], 'confidence': 1.0},
            'Client_077':                 {'aliases': [], 'confidence': 1.0},
            'Client_078':                 {'aliases': [], 'confidence': 1.0},
            'Client_039':              {'aliases': [], 'confidence': 1.0},
            'Client_040':              {'aliases': [], 'confidence': 1.0},
            'Client_003':     {'aliases': [], 'confidence': 1.0},
            'Client_041':              {'aliases': ['Shanti Enterrises'], 'confidence': 1.0},
            'Client_042':              {'aliases': [], 'confidence': 1.0},
            'Client_022':            {'aliases': [], 'confidence': 1.0},
            'Client_143':                        {'aliases': [], 'confidence': 1.0},
            'Client_065':                {'aliases': [], 'confidence': 1.0},
            'Client_117':                    {'aliases': [], 'confidence': 1.0},
            'Client_134':                      {'aliases': [], 'confidence': 1.0},
            'Client_144':                        {'aliases': [], 'confidence': 1.0},
            'Client_066':                {'aliases': [], 'confidence': 1.0},
            'Client_079':                 {'aliases': ['The Garden Room.'], 'confidence': 1.0},
            'Client_005':       {'aliases': [], 'confidence': 1.0},
            'Client_043':              {'aliases': [], 'confidence': 1.0},
            'Client_080':                 {'aliases': ['V.K.Enterprises'], 'confidence': 1.0},
            'Client_102':                   {'aliases': [], 'confidence': 1.0},
            'Client_118':                    {'aliases': [], 'confidence': 1.0},
            'Client_081':                 {'aliases': [], 'confidence': 1.0},
            'Client_013':          {'aliases': [], 'confidence': 1.0},
            'Client_093':                  {'aliases': [], 'confidence': 1.0},
        }
    def _init_products(self):
        self.products = {
            'Client_152':                       {'aliases': ['SUNFAB', 'Sun FAB', 'FAB', 'Fabric Pergola'], 'confidence': 1.0},
            'Client_153':                       {'aliases': ['SUNZIP', 'Sun ZIP', 'ZIP', 'Zip Blind'], 'confidence': 1.0},
            'Client_139':                    {'aliases': ['SUNLOUVER', 'Sun Louver', 'Louver'], 'confidence': 1.0},
            'Client_148':                      {'aliases': ['SUNLITE', 'Sun Lite', 'Lite'], 'confidence': 0.9},
            'Client_149':                      {'aliases': ['STARPOD', 'Star Pod', 'Pod'], 'confidence': 0.95},
            'Client_094':               {'aliases': ['MONSOON BLIND', 'MONSOON BLINDS', 'Monsoon Blind'], 'confidence': 1.0},
            'Client_016':        {'aliases': ['SHEARFAB', 'SHADE SAIL', 'Shade Sail', 'ShearFab', 'Sail Shade'], 'confidence': 1.0},
            'Client_082':              {'aliases': ['SUNMAX', 'GAZEBO', 'SunMax', 'Gazebo'], 'confidence': 1.0},
            'Client_008':      {'aliases': ['SKYLIGHT', 'Skylight', 'Skylight Blind'], 'confidence': 1.0},
            'Client_002': {'aliases': ['WEATHER BLIND', 'EXTERIOR BLIND', 'Weather Blind', 'Exterior Blind', 'MANUAL EXTERIOR', 'MANUAL WEATHER'], 'confidence': 1.0},
        }

    def _init_cities(self):
        self.city_map = {}
        cities = {
            'Mumbai':      ['Mumbai', 'Bombay', 'Mum'],
            'Bengaluru':   ['Bengaluru', 'Bangalore', 'BLR'],
            'Delhi':       ['Delhi', 'New Delhi', 'NCR'],
            'Chennai':     ['Chennai', 'Madras'],
            'Kolkata':     ['Kolkata', 'Calcutta'],
            'Pune':        ['Pune', 'Poona'],
            'Hyderabad':   ['Hyderabad', 'Secunderabad'],
            'Surat':       ['Surat'],
            'Ahmedabad':   ['Ahmedabad'],
            'Vadodara':    ['Vadodara', 'Baroda'],
            'Coimbatore':  ['Coimbatore', 'Kovai'],
            'Lonavala':    ['Lonavala', 'Lonavla'],
            'Kharghar':    ['Kharghar'],
            'Rajkot':      ['Rajkot'],
            'Bhilai':      ['Bhilai'],
            'Amritsar':    ['Amritsar'],
            'Pawagadh':    ['Pawagadh', 'Pavagadh'],
            'Nashik':      ['Nashik', 'Nasik'],
            'Baroda':      ['Baroda'],
        }
        for canonical, aliases in cities.items():
            for alias in aliases:
                self.city_map[alias.lower()] = canonical

    def _init_salespersons(self):
        self.salespersons = {
            'Client_095':  {'aliases': ['Samir Desai', 'Samir', 'S. Desai', 'S.S. Desai'], 'confidence': 1.0},
            'Client_096':  {'aliases': ['Surendra', 'S. Jedhe'], 'confidence': 1.0},
            'Client_103':   {'aliases': ['Chetna', 'C. Bhanot', 'Chetna Banot'], 'confidence': 1.0},
            'Client_135':      {'aliases': ['Ravi', 'R. Mehta'], 'confidence': 1.0},
            'Client_104':   {'aliases': ['Rajesh', 'R. Sharma'], 'confidence': 1.0},
            'Client_140':       {'aliases': ['Om', 'O. Tiwari'], 'confidence': 1.0},
            'Client_105':   {'aliases': ['Dharmesh', 'D. Vora'], 'confidence': 1.0},
            'Client_128':     {'aliases': ['Meena', 'M. Joshi'], 'confidence': 1.0},
            'Client_119':    {'aliases': ['Anand', 'A. Pathak'], 'confidence': 1.0},
            'Client_120':    {'aliases': ['Akash', 'A. Bansal'], 'confidence': 1.0},
            'Client_106':   {'aliases': ['Romal', 'R. Jaiswal'], 'confidence': 1.0},
            'Client_107':   {'aliases': ['Girish', 'G. Parikh'], 'confidence': 1.0},
        }

    def normalize_client(self, name: str) -> Tuple[str, float]:
        """Normalize client name against master list"""
        if not name:
            return None, 0

        name_lower = name.lower().strip()

        for canonical, info in self.clients.items():
            if name_lower == canonical.lower():
                return canonical, info['confidence']
            for alias in info['aliases']:
                if name_lower == alias.lower():
                    return canonical, info['confidence'] * 0.95

        # No master-list match — return raw extracted name rather than wrong fuzzy mapping
        return (name, 0.55)

    def normalize_product(self, product: str) -> Tuple[str, float]:
        """Normalize product name"""
        if not product:
            return None, 0

        product_upper = product.upper().strip()

        for canonical, info in self.products.items():
            if product_upper == canonical.upper():
                return canonical, info['confidence']
            for alias in info['aliases']:
                if product_upper == alias.upper():
                    return canonical, info['confidence'] * 0.95

        # Check if product name is contained in text
        for canonical, info in self.products.items():
            if canonical.upper() in product_upper:
                return canonical, info['confidence'] * 0.8

        return product, 0.3

    def normalize_city(self, city: str) -> Tuple[str, float]:
        """Normalize city name"""
        if not city:
            return None, 0

        city_lower = city.lower().strip()

        if city_lower in self.city_map:
            return self.city_map[city_lower], 0.95

        return city, 0.5

    def normalize_salesperson(self, name: str) -> Tuple[str, float]:
        """Normalize salesperson name"""
        if not name:
            return None, 0

        name_lower = name.lower().strip()

        for canonical, info in self.salespersons.items():
            if name_lower == canonical.lower():
                return canonical, info['confidence']
            for alias in info['aliases']:
                if name_lower == alias.lower():
                    return canonical, info['confidence'] * 0.9

        return name, 0.4

master = MasterData()
print("✅ Master data initialized")

In [ ]:
# CELL 4: Document Text Extractor (Handles DOCX, PDF, Images, and Excel files)
from docx import Document
import pdfplumber
from PIL import Image
import pytesseract

class DocumentExtractor:
    """Extract text from various document formats"""

    SUPPORTED_EXTS = {'.pdf', '.docx', '.jpg', '.jpeg', '.png', '.xlsx', '.xls'}

    def extract(self, filepath: Path) -> Tuple[str, str]:
        """
        Extract text from document.
        Returns: (text, method) where method is 'direct', 'ocr', or 'failed'
        """
        ext = filepath.suffix.lower()

        if ext == '.pdf':
            return self._extract_pdf(filepath)
        elif ext == '.docx':
            return self._extract_docx(filepath)
        elif ext in {'.xlsx', '.xls'}:
            return self._extract_excel(filepath)
        elif ext in {'.jpg', '.jpeg', '.png'}:
            return self._extract_image(filepath)
        else:
            return "", "failed"

    def _extract_pdf(self, filepath: Path) -> Tuple[str, str]:
        try:
            text = ""
            with pdfplumber.open(filepath) as pdf:
                for page in pdf.pages[:10]:  # First 10 pages max
                    page_text = page.extract_text() or ""
                    text += page_text
            return text, "direct" if len(text) > 100 else "failed"
        except Exception as e:
            return "", "failed"

    def _extract_docx(self, filepath: Path) -> Tuple[str, str]:
        try:
            doc = Document(filepath)
            all_text = []

            # Extract paragraphs
            for para in doc.paragraphs:
                if para.text.strip():
                    all_text.append(para.text)

            # Extract tables (critical for amount data)
            for table in doc.tables:
                for row in table.rows:
                    row_text = []
                    for cell in row.cells:
                        if cell.text.strip():
                            row_text.append(cell.text.strip())
                    if row_text:
                        all_text.append(" | ".join(row_text))

            return "\n".join(all_text), "direct"
        except Exception as e:
            return "", "failed"

    def _extract_excel(self, filepath: Path) -> Tuple[str, str]:
        """
        Extract text from Excel files (.xlsx, .xls).
        Uses row-iteration to reliably find TOTAL / GRAND TOTAL / AMOUNT PAYABLE rows
        and adjacent numeric values, instead of converting to string and running regex.
        """
        try:
            import pandas as pd

            all_text = []
            found_amounts = []

            ext = filepath.suffix.lower()
            engine = 'xlrd' if ext == '.xls' else 'openpyxl'

            excel_file = pd.ExcelFile(filepath, engine=engine)

            for sheet_name in excel_file.sheet_names:
                df = pd.read_excel(filepath, sheet_name=sheet_name, engine=engine, header=None)

                all_text.append(f"\n=== SHEET: {sheet_name} ===\n")

                # Row-by-row scan (Fix 5)
                for row_idx in range(len(df)):
                    row_vals = df.iloc[row_idx].tolist()
                    row_strs = [str(v).strip().upper() for v in row_vals]

                    # Check if any cell in this row is a total-label
                    is_total_row = any(
                        lbl in cell
                        for cell in row_strs
                        for lbl in ('TOTAL AMOUNT', 'GRAND TOTAL', 'AMOUNT PAYABLE', 'TOTAL')
                    )

                    if is_total_row:
                        # Scan ALL cells in the row for a numeric value > 10,000
                        for val in row_vals:
                            try:
                                num = float(str(val).replace(',', '').replace('₹', '').strip())
                                if num > 10_000:
                                    found_amounts.append(num)
                            except (ValueError, TypeError):
                                pass

                    # Also emit row as pipe-joined text for field extraction
                    row_text = ' | '.join(str(v) for v in row_vals if str(v).strip() not in ('', 'nan'))
                    if row_text:
                        all_text.append(row_text)

            # Inject the best found amount directly into the text so regex can pick it up
            if found_amounts:
                best = max(found_amounts)
                all_text.append(f"\nTOTAL AMOUNT: {best:,.0f}")

            combined_text = "\n".join(all_text)
            return (combined_text, "direct") if len(combined_text) > 100 else ("", "failed")

        except ImportError:
            import subprocess, sys
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'openpyxl', 'xlrd', '-q'])
            return self._extract_excel(filepath)
        except Exception as e:
            return "", "failed"

    def _extract_image(self, filepath: Path) -> Tuple[str, str]:
        try:
            img = Image.open(filepath).convert('RGB')
            text = pytesseract.image_to_string(img, config='--psm 6')
            return text, "ocr"
        except Exception as e:
            return "", "failed"

extractor = DocumentExtractor()
print("✅ Document extractor ready (supports PDF, DOCX, Excel, Images)")

In [ ]:
# CELL 5: Complete Regex Extractor - FIXED Product Detection

class RegexExtractor:
    """Complete extraction for all quotation fields"""

    def extract_all(self, text: str, filename: str) -> Dict[str, ExtractedField]:
        results = {}

        # Extract from filename
        self._extract_from_filename(filename, results)

        # Extract ALL fields from text
        self._extract_client(text, results)
        self._extract_quotation_number(text, results)
        self._extract_date(text, results)
        self._extract_product(text, results)  # FIXED - Priority to real products
        self._extract_salesperson(text, results)
        self._extract_city(text, results)
        self._extract_total_amount(text, results)
        self._extract_gst(text, results)
        self._extract_subtotal(text, results)
        self._extract_area(text, results)
        self._extract_rate(text, results)
        self._extract_discount(text, results)

        # Remove any non-ExtractedField values
        results = {k: v for k, v in results.items() if isinstance(v, ExtractedField)}

        return results

    def _extract_from_filename(self, filename: str, results: Dict):
        """Extract from filename pattern: '101 - Client - City - Product'"""
        match = re.search(r'^(\d+)', filename)
        if match:
            results['quotation_no'] = ExtractedField(
                value=f"SIMPL/{match.group(1)}",
                confidence=0.95,
                source="filename"
            )

        parts = filename.replace('.docx', '').replace('.pdf', '').split(' - ')
        if len(parts) >= 2:
            client_name = parts[1].strip()
            if len(client_name) > 2:
                results['_filename_client'] = client_name

        if len(parts) >= 3:
            city_name = parts[2].strip()
            normalized, conf = master.normalize_city(city_name)
            if conf > 0.5:
                results['city'] = ExtractedField(
                    value=normalized,
                    confidence=conf * 0.80,
                    source="filename"
                )

        # FIXED: Better product extraction from filename
        if len(parts) >= 4:
            product_name = parts[3].strip().upper()

            # Check for actual products first
            if 'SUNFAB' in product_name or 'FABRIC PERGOLA' in product_name or 'PERGOLA' in product_name:
                results['product_type'] = ExtractedField(value='Client_009', confidence=0.95, source="filename")
            elif 'SUNZIP' in product_name or 'ZIP BLIND' in product_name or 'ZIP SCREEN' in product_name:
                results['product_type'] = ExtractedField(value='Client_004', confidence=0.95, source="filename")
            elif 'SUNLOUVER' in product_name or 'LOUVER' in product_name:
                results['product_type'] = ExtractedField(value='Client_001', confidence=0.95, source="filename")
            elif 'STARPOD' in product_name or 'STAR POD' in product_name or 'DOME' in product_name:
                results['product_type'] = ExtractedField(value='Client_006', confidence=0.95, source="filename")
            elif 'SUNLITE' in product_name or 'GAZEBO' in product_name:
                results['product_type'] = ExtractedField(value='Client_067', confidence=0.95, source="filename")
            elif 'SUNMAX' in product_name:
                results['product_type'] = ExtractedField(value='Client_082', confidence=0.95, source="filename")

    def _extract_client(self, text: str, results: Dict):
        if 'client' in results and isinstance(results.get('client'), ExtractedField):
            return

        first_lines = text.split('\n')[:15]
        address_patterns = [
            r'To\s*[:\-]\s*(.{3,80})',
            r'Attn\s*[:\-]\s*(.{3,80})',
            r'M/s\.?\s+(.{3,80})',
            r'Dear\s+(?:Mr\.|Ms\.|Mrs\.|Dr\.)?\s*([A-Za-z][A-Za-z\s\.&]{2,60})',
        ]
        for line in first_lines:
            for pattern in address_patterns:
                match = re.search(pattern, line.strip(), re.IGNORECASE)
                if match:
                    client = match.group(1).strip().rstrip(',')
                    client = re.sub(r'\s+', ' ', client)
                    if 3 < len(client) < 100:
                        normalized, conf = master.normalize_client(client)
                        results['client'] = ExtractedField(
                            value=normalized,
                            confidence=conf * 0.90,
                            source="doc_header"
                        )
                        return

        broader_patterns = [
            r'Ms\.\s+([A-Za-z\s\.&]{3,60})',
            r'Mr\.\s+([A-Za-z\s\.&]{3,60})',
            r'Client\s*:\s*([A-Za-z\s\.&]{3,60})',
        ]
        for pattern in broader_patterns:
            match = re.search(pattern, text)
            if match:
                client = match.group(1).strip()
                client = re.sub(r'\s+', ' ', client)
                if 3 < len(client) < 100:
                    normalized, conf = master.normalize_client(client)
                    results['client'] = ExtractedField(
                        value=normalized,
                        confidence=conf * 0.80,
                        source="regex"
                    )
                    return

        if '_filename_client' in results:
            fn_client = results.pop('_filename_client')
            normalized, conf = master.normalize_client(fn_client)
            if normalized:
                results['client'] = ExtractedField(
                    value=normalized,
                    confidence=conf * 0.75,
                    source="filename_fallback"
                )
                return

    def _extract_quotation_number(self, text: str, results: Dict):
        if 'quotation_no' in results and isinstance(results.get('quotation_no'), ExtractedField):
            return

        patterns = [
            r'Quotation No\s*[:\-]\s*([A-Z0-9/\-]+)',
            r'Quote No\s*[:\-]\s*([A-Z0-9/\-]+)',
            r'SIMPL[/\-](\d+)',
            r'Q-(\d{5,})',
            r'QT[/\-](\d+)',
        ]

        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                qno = match.group(1) if match.groups() else match.group(0)
                if 'SIMPL' not in qno and qno.isdigit():
                    qno = f"SIMPL/{qno}"
                results['quotation_no'] = ExtractedField(
                    value=qno,
                    confidence=0.85,
                    source="regex"
                )
                return

    # ============================================================
    # FIXED: Date Extraction - Multiple formats
    # ============================================================
    def _extract_date(self, text: str, results: Dict):
        """Extract date from document - SUPPORTS MULTIPLE FORMATS"""
        if 'date' in results:
            return

        # Comprehensive date patterns
        date_patterns = [
            r'\b(\d{1,2})[-/](Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec|January|February|March|April|May|June|July|August|September|October|November|December)[-/](\d{4})\b',
            r'\b(\d{1,2})[-/](\d{1,2})[-/](\d{4})\b',
            r'\b(\d{4})[-/](\d{1,2})[-/](\d{1,2})\b',
            r'Date\s*:\s*(\d{1,2}[-/][A-Za-z]+[-/]\d{4})',
            r'Date\s*:\s*(\d{1,2}[-/]\d{1,2}[-/]\d{4})',
            r'Quotation Date\s*:\s*(\d{1,2}[-/][A-Za-z]+[-/]\d{4})',
            r'\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec|January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{1,2}),?\s+(\d{4})\b',
            r'\b(\d{1,2})\s+(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec|January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{4})\b',
        ]

        month_map = {
            'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 'May': '05', 'Jun': '06',
            'Jul': '07', 'Aug': '08', 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12',
            'January': '01', 'February': '02', 'March': '03', 'April': '04', 'May': '05', 'June': '06',
            'July': '07', 'August': '08', 'September': '09', 'October': '10', 'November': '11', 'December': '12'
        }

        for pattern in date_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                groups = match.groups()

                if len(groups) == 3:
                    first, second, third = groups

                    if first.isdigit() and second.isalpha():
                        day = f"{int(first):02d}"
                        month = month_map.get(second[:3].capitalize(), '01')
                        year = third
                        formatted_date = f"{day}-{month}-{year}"
                        results['date'] = ExtractedField(value=formatted_date, confidence=0.90, source="regex")
                        return

                    elif first.isalpha() and second.isdigit():
                        month = month_map.get(first[:3].capitalize(), '01')
                        day = f"{int(second):02d}"
                        year = third
                        formatted_date = f"{day}-{month}-{year}"
                        results['date'] = ExtractedField(value=formatted_date, confidence=0.90, source="regex")
                        return

                    elif first.isdigit() and second.isdigit() and third.isdigit():
                        if len(first) == 4:
                            year, month, day = first, f"{int(second):02d}", f"{int(third):02d}"
                            formatted_date = f"{day}-{month}-{year}"
                        else:
                            day, month, year = f"{int(first):02d}", f"{int(second):02d}", third
                            formatted_date = f"{day}-{month}-{year}"
                        results['date'] = ExtractedField(value=formatted_date, confidence=0.90, source="regex")
                        return

                elif len(groups) == 1:
                    date_str = groups[0]
                    for fmt in ['%d-%b-%Y', '%d-%B-%Y', '%d/%m/%Y', '%d-%m-%Y']:
                        try:
                            parsed = datetime.strptime(date_str, fmt)
                            results['date'] = ExtractedField(
                                value=parsed.strftime('%d-%m-%Y'),
                                confidence=0.85,
                                source="regex"
                            )
                            return
                        except:
                            continue

        filename = results.get('_source_file', '')
        if filename:
            file_match = re.search(r'(\d{1,2})[.-](\d{1,2})[.-](\d{4})', filename)
            if file_match:
                day, month, year = file_match.groups()
                formatted_date = f"{int(day):02d}-{int(month):02d}-{year}"
                results['date'] = ExtractedField(value=formatted_date, confidence=0.70, source="filename")
                return

    # ============================================================
    # FIXED: Product Detection - NO "Blackout" default
    # ============================================================
    def _extract_product(self, text: str, results: Dict):
        """Extract product type - Priority to actual products, NO default to Blackout"""
        if 'product_type' in results and isinstance(results.get('product_type'), ExtractedField):
            return

        text_lower = text.lower()
        text_upper = text.upper()

        # Comprehensive product mapping with keywords and confidence
        product_keywords = {
            'Client_009': {
                'keywords': ['sunfab', 'sun fab', 'fabric pergola', 'retractable pergola',
                            'motorised pergola', 'motorized pergola', 'fabric roof',
                            'fabric canopy', 'pergola', 'fabric structure'],
                'exclude': ['louver', 'zip', 'blind', 'louvre', 'glass', 'polycarbonate', 'gazebo']
            },
            'Client_004': {
                'keywords': ['sunzip', 'sun zip', 'zip blind', 'zip screen', 'motorised zip',
                            'motorized zip', 'zip track', 'zip shade', 'ziptrack', 'zip roller',
                            'zip system', 'zip blind system'],
                'exclude': ['pergola', 'louver']
            },
            'Client_001': {
                'keywords': ['sunlouver', 'sun louver', 'louver pergola', 'louvre pergola',
                            'aluminium pergola', 'aluminum pergola', 'aluminium louvre',
                            'louvred roof', 'louvered roof', 'motorised louver', 'aluminum louver',
                            'louver system', 'bioclimatic pergola'],
                'exclude': ['fabric', 'zip', 'blind']
            },
            'Client_067': {
                'keywords': ['sunlite', 'sun lite', 'polycarbonate', 'poly pergola', 'pc sheet',
                            'polycarbonate sheet', 'polycarbonate roof', 'poly roof', 'gazebo'],
                'exclude': ['fabric', 'louver']
            },
            'Client_082': {
                'keywords': ['sunmax', 'sun max', 'glass pergola', 'glass roof', 'glass canopy',
                            'glass louvre', 'glass structure', 'glass gazebo'],
                'exclude': ['fabric', 'polycarbonate']
            },
            'Client_006': {
                'keywords': ['starpod', 'star pod', 'dome', 'gazebo dome', 'pod structure',
                            'garden pod', 'outdoor pod', 'garden gazebo', 'dome structure'],
                'exclude': []
            },
            'Client_002': {
                'keywords': ['weather blind', 'exterior blind', 'manual weather', 'weather blind system',
                            'exterior weather blind', 'manual exterior'],
                'exclude': []
            },
            'Client_016': {
                'keywords': ['shearfab', 'shade sail', 'sail shade', 'shear fab', 'sail shade system'],
                'exclude': []
            }
        }

        # First priority: Check filename for product indicators
        filename = results.get('_source_file', '')
        if filename:
            filename_upper = filename.upper()
            for product, info in product_keywords.items():
                for keyword in info['keywords']:
                    if keyword.upper() in filename_upper:
                        # Check exclusion
                        excluded = False
                        for ex in info.get('exclude', []):
                            if ex.upper() in filename_upper:
                                excluded = True
                                break
                        if not excluded:
                            results['product_type'] = ExtractedField(
                                value=product,
                                confidence=0.90,
                                source="filename_product"
                            )
                            return

        # Second priority: Check document text
        best_product = None
        best_confidence = 0
        best_keyword = None

        for product, info in product_keywords.items():
            for keyword in info['keywords']:
                if keyword in text_lower:
                    # Check exclusions
                    excluded = False
                    for ex in info.get('exclude', []):
                        if ex in text_lower:
                            excluded = True
                            break

                    if not excluded:
                        # Higher confidence for exact matches
                        confidence = 0.85
                        if keyword == product.lower():
                            confidence = 0.95
                        elif keyword in ['sunfab', 'sunzip', 'sunlouver']:
                            confidence = 0.90

                        if confidence > best_confidence:
                            best_confidence = confidence
                            best_product = product
                            best_keyword = keyword

        if best_product:
            results['product_type'] = ExtractedField(
                value=best_product,
                confidence=best_confidence,
                source="regex"
            )
            return

        # Third priority: Check for product in tables
        if 'area' in results and results['area']:
            if 'louver' in text_lower or 'aluminium' in text_lower:
                results['product_type'] = ExtractedField(
                    value='Client_001',
                    confidence=0.70,
                    source="area_context"
                )
                return
            elif 'fabric' in text_lower or 'pergola' in text_lower:
                results['product_type'] = ExtractedField(
                    value='Client_009',
                    confidence=0.70,
                    source="area_context"
                )
                return

        # IMPORTANT: Do NOT default to "Blackout"
        # If no product found, leave as None (will show blank in report)
        pass

    def _extract_salesperson(self, text: str, results: Dict):
        """Extract salesperson name - NO default, only when found"""
        if 'salesperson' in results:
            return

        # Valid salesperson names
        VALID_SALESPEOPLE = {
            'Client_095': ['Samir Desai', 'Client_095', 'Samir S Desai', 'Sameer Desai', 'Samir', 'S. Desai'],
            'Client_096': ['Client_096', 'Surendra', 'S. Jedhe'],
            'Client_103': ['Client_103', 'Chetna Banot', 'Chetna', 'C. Bhanot'],
            'Client_135': ['Client_135', 'Ravi', 'R. Mehta'],
            'Client_104': ['Client_104', 'Rajesh', 'R. Sharma', 'Rajessh Sharma'],
            'Client_140': ['Client_140', 'Om', 'O. Tiwari', 'Omprakash Tiwari'],
            'Client_105': ['Client_105', 'Dharmesh', 'D. Vora'],
            'Client_128': ['Client_128', 'Meena', 'M. Joshi'],
            'Client_119': ['Client_119', 'Anand', 'A. Pathak'],
            'Client_120': ['Client_120', 'Akash', 'A. Bansal'],
            'Client_106': ['Client_106', 'Romal', 'R. Jaiswal'],
            'Client_107': ['Client_107', 'Girish', 'G. Parikh'],
        }

        INVALID_SP_WORDS = {
            'structure', 'beams', 'guideways', 'pipes', 'ms pipes', 'above price is ex',
            'ex', 'with', 'steel', 'aluminium', 'fabric', 'motorised', 'retractable',
            'pergola', 'sunfab', 'sunzip', 'sunlouver', 'starpod', 'hello', 'dear',
            'sir', 'madam', 'thanks', 'regards', 'kind attn', 'marketing', 'sales',
            'team', 'company', 'corporation', 'pvt', 'ltd', 'llp', 'inc', '& co',
            'associates', 'enterprises', 'industries', 'infra', 'builders', 'developers',
            'prepared', 'issued', 'quoted', 'executive', 'representative', 'rep',
            'manager', 'director', 'ceo', 'founder', 'owner', 'partner', 'warm', 'regards',
            'tarrannum', 'mehra', 'stellar', 'omprakash', 'tiwari', 'offer', 'sunfab'
        }

        def is_valid_sp_name(name):
            if not name:
                return False
            name_lower = name.lower().strip()
            if name_lower in INVALID_SP_WORDS:
                return False
            if len(name) < 3:
                return False
            return True

        sp_found = None

        # Priority 1: "Prepared by:" labels
        label_patterns = [
            (r'Prepared\s+by\s*:\s*([A-Z][a-z]+(?:\s+[A-Z][a-z\.]+){0,2})', 0.95),
            (r'Sales\s+Executive\s*:\s*([A-Z][a-z]+(?:\s+[A-Z][a-z\.]+){0,2})', 0.95),
            (r'Quoted\s+by\s*:\s*([A-Z][a-z]+(?:\s+[A-Z][a-z\.]+){0,2})', 0.95),
        ]

        for pattern, conf in label_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                candidate = match.group(1).strip()
                if is_valid_sp_name(candidate):
                    for canonical, aliases in VALID_SALESPEOPLE.items():
                        if candidate.lower() == canonical.lower() or any(alias.lower() == candidate.lower() for alias in aliases):
                            sp_found = canonical
                            break
                    if not sp_found and len(candidate.split()) <= 3:
                        sp_found = candidate.title()
                    if sp_found:
                        results['salesperson'] = ExtractedField(
                            value=sp_found,
                            confidence=conf,
                            source="label_pattern"
                        )
                        return

        # Priority 2: Look in signature area
        tail_lines = text.split('\n')[-30:]
        tail_text = '\n'.join(tail_lines)

        for canonical, aliases in VALID_SALESPEOPLE.items():
            for name in [canonical] + aliases:
                if re.search(r'\b' + re.escape(name) + r'\b', tail_text, re.IGNORECASE):
                    results['salesperson'] = ExtractedField(
                        value=canonical,
                        confidence=0.90,
                        source="signature_match"
                    )
                    return

        # Priority 3: Direct match anywhere
        for canonical, aliases in VALID_SALESPEOPLE.items():
            for name in [canonical] + aliases:
                if re.search(r'\b' + re.escape(name) + r'\b', text, re.IGNORECASE):
                    results['salesperson'] = ExtractedField(
                        value=canonical,
                        confidence=0.80,
                        source="document_match"
                    )
                    return

        # No salesperson found - leave None
        pass

    def _extract_city(self, text: str, results: Dict):
        if 'city' in results and isinstance(results.get('city'), ExtractedField):
            return

        cities = list({
            'Mumbai', 'Delhi', 'Bengaluru', 'Bangalore', 'Chennai', 'Kolkata',
            'Pune', 'Hyderabad', 'Ahmedabad', 'Surat', 'Vadodara', 'Baroda',
            'Gurgaon', 'Noida', 'Lucknow', 'Jaipur', 'Indore', 'Nagpur',
            'Bhopal', 'Ludhiana', 'Agra', 'Meerut', 'Nashik', 'Nasik',
            'Coimbatore', 'Lonavala', 'Lonavla', 'Kharghar', 'Rajkot',
            'Bhilai', 'Amritsar', 'Pawagadh', 'Pavagadh', 'Baroda',
        })

        def find_city_in_text(haystack: str):
            for city in cities:
                if re.search(r'\b' + re.escape(city) + r'\b', haystack, re.IGNORECASE):
                    normalized, conf = master.normalize_city(city)
                    return normalized, conf
            return None, 0

        site_patterns = [
            r'Site\s+at[:\\s]+(.{5,120})',
            r'Project\s+Location[:\\s]+(.{5,120})',
            r'Delivery\s+at[:\\s]+(.{5,120})',
            r'Installation\s+at[:\\s]+(.{5,120})',
        ]
        for pattern in site_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                context = match.group(1)
                city, conf = find_city_in_text(context)
                if city:
                    results['city'] = ExtractedField(value=city, confidence=conf * 0.95, source="site_address")
                    return

        city, conf = find_city_in_text(text)
        if city:
            results['city'] = ExtractedField(value=city, confidence=conf * 0.70, source="regex_fallback")

    def _extract_total_amount(self, text: str, results: Dict):
        if 'total_amount' in results:
            return

        lines = text.split('\n')

        for i, line in enumerate(lines):
            if 'TOTAL AMOUNT' in line.upper():
                numbers = re.findall(r'(\d{1,2},\d{2,3},\d{3})', line)
                if numbers:
                    amount_str = numbers[0].replace(',', '')
                    results['total_amount'] = ExtractedField(
                        value=float(amount_str),
                        confidence=0.95,
                        source="total_amount_line"
                    )
                    return

                if i + 1 < len(lines):
                    numbers = re.findall(r'(\d{1,2},\d{2,3},\d{3})', lines[i+1])
                    if numbers:
                        amount_str = numbers[0].replace(',', '')
                        results['total_amount'] = ExtractedField(
                            value=float(amount_str),
                            confidence=0.95,
                            source="total_amount_line"
                        )
                        return

        for line in lines:
            if 'GRAND TOTAL' in line.upper():
                numbers = re.findall(r'(\d{1,2},\d{2,3},\d{3})', line)
                if numbers:
                    amount_str = numbers[0].replace(',', '')
                    results['total_amount'] = ExtractedField(
                        value=float(amount_str),
                        confidence=0.90,
                        source="grand_total"
                    )
                    return

        match = re.search(r'[₹]\s*(\d{1,2},\d{2,3},\d{3})', text)
        if match:
            amount_str = match.group(1).replace(',', '')
            results['total_amount'] = ExtractedField(
                value=float(amount_str),
                confidence=0.80,
                source="rupee_symbol"
            )
            return

        all_numbers = re.findall(r'(\d{1,2},\d{2,3},\d{3})', text)
        if all_numbers:
            amounts = [float(n.replace(',', '')) for n in all_numbers]
            max_amount = max(amounts)
            if max_amount > 10000:
                results['total_amount'] = ExtractedField(
                    value=max_amount,
                    confidence=0.70,
                    source="max_number_fallback"
                )

    def _extract_gst(self, text: str, results: Dict):
        if 'gst' in results:
            return

        lines = text.split('\n')

        for line in lines:
            if 'GST' in line.upper():
                if '18' in line:
                    numbers = re.findall(r'(\d{1,2},\d{2,3},\d{3})', line)
                    if numbers:
                        gst_str = numbers[0].replace(',', '')
                        results['gst'] = ExtractedField(
                            value=float(gst_str),
                            confidence=0.90,
                            source="gst_line"
                        )
                        return

                match = re.search(r'GST\s*(\d{1,2})%\s*[:\\-]?\s*[₹]?\s*(\d{1,2},\d{2,3},\d{3})', line, re.IGNORECASE)
                if match:
                    gst_str = match.group(2).replace(',', '')
                    results['gst'] = ExtractedField(
                        value=float(gst_str),
                        confidence=0.85,
                        source="gst_percentage"
                    )
                    return

    def _extract_subtotal(self, text: str, results: Dict):
        if 'subtotal' in results:
            return

        lines = text.split('\n')

        for line in lines:
            if line.strip().startswith('Total') and 'AMOUNT' not in line.upper():
                numbers = re.findall(r'(\d{1,2},\d{2,3},\d{3})', line)
                if numbers:
                    subtotal_str = numbers[0].replace(',', '')
                    if 'total_amount' not in results or float(subtotal_str) < results['total_amount'].value:
                        results['subtotal'] = ExtractedField(
                            value=float(subtotal_str),
                            confidence=0.85,
                            source="subtotal_line"
                        )
                        return

        for line in lines:
            if 'SUB TOTAL' in line.upper():
                numbers = re.findall(r'(\d{1,2},\d{2,3},\d{3})', line)
                if numbers:
                    subtotal_str = numbers[0].replace(',', '')
                    results['subtotal'] = ExtractedField(
                        value=float(subtotal_str),
                        confidence=0.85,
                        source="subtotal_label"
                    )
                    return

    def _extract_area(self, text: str, results: Dict):
        if 'area_sqft' in results:
            return

        patterns = [
            r'SQFT\s*\|?\s*(\d{3,5})',
            r'SQ\.FT\.\s*[:]?\s*(\d{3,5})',
            r'Area\s*[:]\s*(\d{3,5})\s*SQ',
            r'Total Area\s*[:]\s*(\d{3,5})',
        ]

        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                area = match.group(1).replace(',', '')
                results['area_sqft'] = ExtractedField(
                    value=float(area),
                    confidence=0.75,
                    source="regex"
                )
                return

    def _extract_rate(self, text: str, results: Dict):
        if 'rate' in results:
            return

        patterns = [
            r'RATE\s*PER\s*SQ\.FT\.\s*[:]?\s*[₹]\s*(\d{1,3}(?:,\d{3})*)',
            r'Rate\s*:\s*[₹]\s*(\d{1,3}(?:,\d{3})*)',
        ]

        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                rate_str = match.group(1).replace(',', '')
                results['rate'] = ExtractedField(
                    value=float(rate_str),
                    confidence=0.75,
                    source="regex"
                )
                return

    def _extract_discount(self, text: str, results: Dict):
        if 'discount' in results:
            return

        patterns = [
            r'Discount\s*[:]?\s*[₹]?\s*(\d{1,2},\d{2,3},\d{3})',
            r'Discount\s*(\d+)%\s*[:]?\s*[₹]?\s*(\d{1,2},\d{2,3},\d{3})',
        ]

        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                discount_group = match.group(2) if match.lastindex == 2 else match.group(1)
                discount_str = discount_group.replace(',', '')
                results['discount'] = ExtractedField(
                    value=float(discount_str),
                    confidence=0.80,
                    source="regex"
                )
                return

regex_extractor = RegexExtractor()
print("✅ Complete Regex Extractor ready - FIXED Product Detection (No 'Blackout' default)")
print("   - Products: SunFAB, SunZIP, SunLouver, StarPod, SunLite, SunMax")
print("   - Salesperson: Only extracted when found (no default)")
print("   - Dates: Multiple formats supported")

In [ ]:
# CELL 6: Correction Database (Learning System)
class CorrectionDB:
    """Store and apply user corrections for continuous improvement"""

    def __init__(self, db_path: Path):
        self.db_path = db_path
        self.conn = sqlite3.connect(str(db_path))
        self._init_tables()
        self._load_cache()

    def _init_tables(self):
        cursor = self.conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS corrections (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                field_type TEXT NOT NULL,
                original TEXT NOT NULL,
                corrected TEXT NOT NULL,
                applied_count INTEGER DEFAULT 1,
                created_at TEXT NOT NULL,
                UNIQUE(field_type, original, corrected)
            )
        ''')
        self.conn.commit()

    def _load_cache(self):
        """Load corrections into memory cache"""
        self.cache = {}
        cursor = self.conn.cursor()
        cursor.execute('SELECT field_type, original, corrected, applied_count FROM corrections')

        for field_type, original, corrected, count in cursor.fetchall():
            if field_type not in self.cache:
                self.cache[field_type] = {}
            self.cache[field_type][original.lower()] = {
                'corrected': corrected,
                'confidence': min(0.95, 0.70 + (count * 0.02))
            }

    def add(self, field_type: str, original: str, corrected: str) -> bool:
        """Add or update a correction"""
        if not original or not corrected or original == corrected:
            return False

        cursor = self.conn.cursor()
        now = datetime.now().isoformat()

        try:
            cursor.execute('''
                INSERT INTO corrections (field_type, original, corrected, applied_count, created_at)
                VALUES (?, ?, ?, 1, ?)
                ON CONFLICT(field_type, original, corrected) DO UPDATE SET
                    applied_count = applied_count + 1,
                    created_at = ?
            ''', (field_type, original, corrected, now, now))

            self.conn.commit()

            # Update cache
            if field_type not in self.cache:
                self.cache[field_type] = {}

            cursor.execute('SELECT applied_count FROM corrections WHERE field_type=? AND original=?',
                          (field_type, original))
            count = cursor.fetchone()[0]

            self.cache[field_type][original.lower()] = {
                'corrected': corrected,
                'confidence': min(0.95, 0.70 + (count * 0.02))
            }

            return True
        except Exception as e:
            return False

    def apply(self, field_type: str, value: str) -> Tuple[str, float, bool]:
        """Apply correction if available"""
        if not value or field_type not in self.cache:
            return value, 0, False

        value_lower = value.lower()
        if value_lower in self.cache[field_type]:
            corr = self.cache[field_type][value_lower]
            return corr['corrected'], corr['confidence'], True

        return value, 0, False

    def get_all(self) -> List[Dict]:
        """Get all corrections for reporting"""
        cursor = self.conn.cursor()
        cursor.execute('SELECT field_type, original, corrected, applied_count FROM corrections')
        return [
            {'field': r[0], 'original': r[1], 'corrected': r[2], 'count': r[3]}
            for r in cursor.fetchall()
        ]

correction_db = CorrectionDB(OUTPUT_DIR / 'corrections.db')
print("✅ Correction database ready")

In [ ]:
# CELL 7: False Positive Detector (FIXED - no inline regex flags)
class FalsePositiveDetector:
    """Prevent extraction of invalid values"""

    def __init__(self):
        # Invalid client patterns - NO inline flags, use re.IGNORECASE in match call
        self.invalid_client_patterns = [
            r'^(mumbai|delhi|bangalore|surat|pune|chennai)$',
            r'^(hello|hi|dear|sir|madam|thanks|regards)$',
            r'^(quotation|quote|proposal|estimate|offer)$',
            r'^(total|amount|price|rate|gst|tax)$',
            r'^(sales|marketing|executive|manager|director)$',
            r'^(pvt|ltd|llp|inc|corp)$',
            r'^\d+$',
            r'^.{1,2}$'
        ]

        # Invalid salesperson patterns
        self.invalid_sp_patterns = [
            r'^(sunmate|simpl|company|corporate)$',
            r'^(quotation|quote|proposal|sales|team)$',
            r'^(customer|client|party)$',
            r'^(thank|regards|sincerely)$',
            r'^\d+$',
            r'^.{1,2}$'
        ]

        # Invalid product patterns
        self.invalid_product_patterns = [
            r'^(mumbai|delhi|bangalore|surat|chennai)$',
            r'^(nath|desai|sunmate|simpl)$',
            r'^\d+$'
        ]

    def validate(self, record: QuotationRecord) -> QuotationRecord:
        """Validate all fields and flag issues"""

        # Validate client
        if record.client and record.client.value:
            client_val = str(record.client.value).strip()
            for pattern in self.invalid_client_patterns:
                if re.match(pattern, client_val, re.IGNORECASE):  # Flag passed here
                    record.needs_review = True
                    record.review_reasons.append(f"Invalid client: '{client_val}'")
                    record.client.confidence *= 0.5
                    break

        # Validate salesperson
        if record.salesperson and record.salesperson.value:
            sp_val = str(record.salesperson.value).strip()
            for pattern in self.invalid_sp_patterns:
                if re.match(pattern, sp_val, re.IGNORECASE):
                    record.needs_review = True
                    record.review_reasons.append(f"Invalid salesperson: '{sp_val}'")
                    record.salesperson.confidence *= 0.5
                    break

        # Validate product
        if record.product_type and record.product_type.value:
            product_val = str(record.product_type.value).strip()
            for pattern in self.invalid_product_patterns:
                if re.match(pattern, product_val, re.IGNORECASE):
                    record.needs_review = True
                    record.review_reasons.append(f"Invalid product: '{product_val}'")
                    record.product_type.confidence *= 0.5
                    break

        # Validate total amount
        if record.total_amount and record.total_amount.value:
            amount = record.total_amount.value
            if amount < config.min_total_amount:
                record.needs_review = True
                record.review_reasons.append(f"Amount too low: {amount}")
                record.total_amount.confidence *= 0.5
            elif amount > config.max_total_amount:
                record.needs_review = True
                record.review_reasons.append(f"Amount unusually high: {amount}")
                record.total_amount.confidence *= 0.8

        return record

fp_detector = FalsePositiveDetector()
print("✅ False positive detector ready (fixed regex patterns)")

In [ ]:
# CELL 8: Main Pipeline (Orchestration) - UPDATED with skip logic
import re
from typing import Optional, List, Dict, Set
from pathlib import Path
import json
import time
import gc
import zipfile

class QuotationPipeline:
    """Main pipeline orchestrating all components"""

    def __init__(self):
        self.processed_files: Set[str] = set()
        self.results: List[QuotationRecord] = []
        self._load_checkpoint()

    def _load_checkpoint(self):
        """Load previously processed files"""
        checkpoint_file = OUTPUT_DIR / 'processed_checkpoint.json'
        if checkpoint_file.exists():
            try:
                with open(checkpoint_file, 'r') as f:
                    self.processed_files = set(json.load(f))
                print(f"📋 Loaded checkpoint: {len(self.processed_files)} files already processed")
            except:
                self.processed_files = set()

    def _save_checkpoint(self):
        """Save processed files"""
        checkpoint_file = OUTPUT_DIR / 'processed_checkpoint.json'
        with open(checkpoint_file, 'w') as f:
            json.dump(list(self.processed_files), f)

    def _is_quotation_file(self, filename: str) -> bool:
        """Check if file is likely a quotation (not a drawing sheet)"""
        name_lower = filename.lower()

        # Skip drawing sheets, floor plans, etc.
        skip_patterns = [
            'drawing', 'sheet', 'floor', 'terrace', 'model', 'swimming pool',
            'g.a.drawing', 'architectural', 'structural', 'plan', 'layout',
            'elevation', 'section', 'detail', 'specification'
        ]

        for pattern in skip_patterns:
            if pattern in name_lower:
                return False

        # Check for quotation indicators
        quotation_indicators = ['quotation', 'quote', 'offer', 'simpl', 'sunfab', 'sunzip']
        for indicator in quotation_indicators:
            if indicator in name_lower:
                return True

        # If filename starts with a number followed by dash and letters, likely a quotation
        if re.match(r'^\d+\s*-\s*[A-Za-z]', filename):
            return True

        return False

    def process_file(self, filepath: Path) -> Optional[QuotationRecord]:
        """Process a single document"""
        filename = filepath.name

        # Skip already processed
        if filename in self.processed_files:
            return None

        # Skip non-quotation files (drawing sheets, etc.)
        if not self._is_quotation_file(filename):
            print(f"  ⏭️ Skipping: {filename[:50]}... (not a quotation)")
            self.processed_files.add(filename)
            self._save_checkpoint()
            return None

        print(f"  📄 Processing: {filename[:60]}...", end=" ")

        # Extract text
        text, method = extractor.extract(filepath)
        if not text or len(text) < 50:
            print("❌ No text extracted")
            self.processed_files.add(filename)
            self._save_checkpoint()
            return None

        # Extract fields
        extracted = regex_extractor.extract_all(text, filename)

        # Build record
        record = QuotationRecord(filename=filename, extraction_method=method)
        record.extracted_at = datetime.now().isoformat()

        # Map extracted fields
        field_map = {
            'quotation_no': 'quotation_no',
            'date': 'date',
            'client': 'client',
            'city': 'city',
            'product_type': 'product_type',
            'total_amount': 'total_amount',
            'gst': 'gst',
            'subtotal': 'subtotal',
            'salesperson': 'salesperson',
            'area_sqft': 'area_sqft',
            'rate': 'rate',
            'discount': 'discount'
        }

        for ext_key, record_key in field_map.items():
            if ext_key in extracted:
                setattr(record, record_key, extracted[ext_key])

        # Apply corrections from database
        for field in ['client', 'salesperson', 'product_type']:
            attr = getattr(record, field)
            if attr and attr.value:
                corrected, boost, applied = correction_db.apply(field, str(attr.value))
                if applied:
                    attr.value = corrected
                    attr.confidence = min(0.95, attr.confidence + boost)

        # Normalize with master data
        if record.client and record.client.value:
            normalized, conf = master.normalize_client(str(record.client.value))
            if normalized != record.client.value:
                record.client.value = normalized
                record.client.confidence = max(record.client.confidence, conf)

        if record.product_type and record.product_type.value:
            normalized, conf = master.normalize_product(str(record.product_type.value))
            if normalized != record.product_type.value:
                record.product_type.value = normalized
                record.product_type.confidence = max(record.product_type.confidence, conf)

        if record.city and record.city.value:
            normalized, conf = master.normalize_city(str(record.city.value))
            if normalized != record.city.value:
                record.city.value = normalized
                record.city.confidence = max(record.city.confidence, conf)

        # Calculate confidence and validate
        record.calculate_confidence()
        record = fp_detector.validate(record)

        # Determine status
        if record.needs_review:
            print(f"⚠️ Review (conf={record.overall_confidence:.0%})")
        else:
            print(f"✅ Done (conf={record.overall_confidence:.0%})")

        # Save
        self.results.append(record)
        self.processed_files.add(filename)
        self._save_checkpoint()

        # Append to JSONL
        jsonl_file = OUTPUT_DIR / 'extractions.jsonl'
        with open(jsonl_file, 'a') as f:
            f.write(json.dumps(record.to_dict()) + '\n')

        return record

    def process_batch(self, files: List[Path]) -> List[QuotationRecord]:
        """Process multiple files in batch"""
        total = len(files)
        print(f"\n📦 Processing {total} file(s)")
        print("=" * 50)

        for i, filepath in enumerate(files, 1):
            self.process_file(filepath)
            time.sleep(config.file_delay)

            if i % 10 == 0:
                gc.collect()
                print(f"  📊 Progress: {i}/{total} files processed")

        print("=" * 50)
        print(f"✅ Processed: {len(self.results)} | Review: {sum(1 for r in self.results if r.needs_review)}")

        return self.results

    def run_from_zip(self, zip_path: Path) -> List[QuotationRecord]:
        """Extract ZIP and process all documents - SORTED by filename"""
        print(f"\n📦 Extracting: {zip_path.name}")

        extract_dir = OUTPUT_DIR / 'extracted'
        extract_dir.mkdir(exist_ok=True)

        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_dir)

        # Find all supported files
        files = []
        for ext in DocumentExtractor.SUPPORTED_EXTS:
            files.extend(extract_dir.rglob(f'*{ext}'))
            files.extend(extract_dir.rglob(f'*{ext.upper()}'))

        # Remove duplicates
        files = list(set(files))

        # Sort files by the number at the beginning of filename (101, 102, etc.)
        def extract_number(filepath):
            match = re.match(r'^(\d+)', filepath.name)
            return int(match.group(1)) if match else 999999

        files.sort(key=extract_number)

        # Filter to only quotation files
        quotation_files = [f for f in files if self._is_quotation_file(f.name)]
        skipped_count = len(files) - len(quotation_files)

        print(f"📁 Total files found: {len(files)}")
        print(f"📋 Quotation files: {len(quotation_files)}")
        print(f"⏭️  Skipped (drawings/other): {skipped_count}")

        if quotation_files:
            print(f"📋 First quotation: {quotation_files[0].name}")
            print(f"📋 Last quotation: {quotation_files[-1].name}")

        return self.process_batch(quotation_files)

pipeline = QuotationPipeline()
print("✅ Pipeline ready - will process quotation files in numerical order (101, 102, 103...)")

In [ ]:
# CELL 8.5: Quick Test on One File
print("="*60)
print("QUICK TEST - Verifying all components")
print("="*60)

# Test extraction on your first file
test_zip = Path(config.input_zip)
extract_dir = OUTPUT_DIR / 'test'
extract_dir.mkdir(exist_ok=True)

# Check if ZIP exists
if not test_zip.exists():
    print(f"❌ ZIP file not found: {test_zip}")
    print("   Please update config.input_zip with correct path")
else:
    with zipfile.ZipFile(test_zip, 'r') as zf:
        # Get first supported file
        supported_exts = {'.pdf', '.docx', '.xlsx', '.xls', '.jpg', '.jpeg', '.png'}
        first_file = None

        for f in zf.namelist():
            if any(f.lower().endswith(ext) for ext in supported_exts):
                first_file = f
                break

        if not first_file:
            print("❌ No supported files found in ZIP")
        else:
            zf.extract(first_file, extract_dir)
            test_path = extract_dir / first_file

            print(f"\n📄 Testing: {first_file}")
            print(f"   File size: {test_path.stat().st_size:,} bytes")

            # Extract text
            text, method = extractor.extract(test_path)
            print(f"   Extraction method: {method}")
            print(f"   Text length: {len(text):,} chars")

            if len(text) > 0:
                print(f"\n📝 First 300 chars of extracted text:")
                print("-" * 40)
                print(text[:300])
                print("-" * 40)

            # Extract fields
            results = regex_extractor.extract_all(text, first_file)

            print(f"\n📊 Extraction Results:")
            print("-" * 40)
            for key, value in results.items():
                status = "✅" if value.confidence > 0.6 else "⚠️"
                print(f"   {status} {key:15s}: {value.value} (conf={value.confidence:.0%})")
            print("-" * 40)

            # Summary
            critical_fields = ['quotation_no', 'client', 'total_amount']
            found = sum(1 for f in critical_fields if f in results)
            print(f"\n📈 Summary: {found}/{len(critical_fields)} critical fields found")

            if found < 2:
                print("\n⚠️ WARNING: Low extraction quality on this file.")
                print("   Check if regex patterns need adjustment for your document format.")
            else:
                print("\n✅ Test passed! Pipeline is working correctly.")

print(f"\n✅ Test complete")

In [ ]:
# CELL 9: Reporting Engine (FIXED - handles empty DataFrames)
import pandas as pd

class ReportGenerator:
    """Generate Excel reports from extraction results"""

    def __init__(self, output_dir: Path):
        self.output_dir = output_dir

    def generate(self, records: List[QuotationRecord]) -> Path:
        """Generate comprehensive Excel report"""

        report_path = self.output_dir / f'Quotation_Report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'

        # Prepare data
        data = []
        for r in records:
            data.append({
                'Filename': r.filename,
                'Quotation No': r.get_value('quotation_no'),
                'Client': r.get_value('client'),
                'City': r.get_value('city'),
                'Product': r.get_value('product_type'),
                'Total Amount (₹)': r.get_value('total_amount'),
                'GST (₹)': r.get_value('gst'),
                'Subtotal (₹)': r.get_value('subtotal'),
                'Area (sq ft)': r.get_value('area_sqft'),
                'Salesperson': r.get_value('salesperson'),
                'Date': r.get_value('date'),
                'Confidence': f"{r.overall_confidence:.0%}" if r.overall_confidence > 0 else "0%",
                'Needs Review': 'Yes' if r.needs_review else 'No',
                'Review Reasons': '; '.join(r.review_reasons) if r.review_reasons else '',
                'Extraction Method': r.extraction_method
            })

        df = pd.DataFrame(data)

        # Only split if DataFrame is not empty
        if len(df) > 0:
            # Split into clean and review (using .loc to avoid KeyError)
            clean_df = df[df['Needs Review'] == 'No'] if 'Needs Review' in df.columns else pd.DataFrame()
            review_df = df[df['Needs Review'] == 'Yes'] if 'Needs Review' in df.columns else pd.DataFrame()
        else:
            clean_df = pd.DataFrame()
            review_df = pd.DataFrame()

        with pd.ExcelWriter(report_path, engine='openpyxl') as writer:
            # Only write sheets if they have data
            if len(clean_df) > 0:
                clean_df.to_excel(writer, sheet_name='Clean Extractions', index=False)

            if len(review_df) > 0:
                review_df.to_excel(writer, sheet_name='Needs Review', index=False)

            # Summary sheet (always include)
            if len(df) > 0:
                avg_confidence = df['Confidence'].str.rstrip('%').astype(float).mean() if len(df) > 0 and 'Confidence' in df.columns else 0
            else:
                avg_confidence = 0

            summary = pd.DataFrame([
                ['Total Documents', len(records)],
                ['Clean Extractions', len(clean_df)],
                ['Needs Review', len(review_df)],
                ['Average Confidence', f"{avg_confidence:.1f}%"] if avg_confidence > 0 else ['Average Confidence', 'N/A'],
                ['Pipeline Version', config.version],
                ['Generated On', datetime.now().strftime("%Y-%m-%d %H:%M:%S")]
            ], columns=['Metric', 'Value'])
            summary.to_excel(writer, sheet_name='Summary', index=False)

            # Corrections sheet
            corrections = correction_db.get_all()
            if corrections:
                corr_df = pd.DataFrame(corrections)
                if len(corr_df) > 0:
                    corr_df.to_excel(writer, sheet_name='Learned Corrections', index=False)

        print(f"\n📊 Report saved: {report_path}")
        return report_path

report_gen = ReportGenerator(OUTPUT_DIR)
print("✅ Report generator ready (fixed for empty DataFrames)")

In [ ]:
# CELL 9.5: RESET AND REPROCESS WITH FIXED SALESPERSON EXTRACTION

import json
from pathlib import Path

# Clear the processed checkpoint
checkpoint_file = OUTPUT_DIR / 'processed_checkpoint.json'
if checkpoint_file.exists():
    checkpoint_file.unlink()
    print("✓ Cleared processed checkpoint")

# Clear the JSONL results file
jsonl_file = OUTPUT_DIR / 'extractions.jsonl'
if jsonl_file.exists():
    jsonl_file.unlink()
    print("✓ Cleared extractions.jsonl")

# Also clear the processed files set in memory
if 'pipeline' in dir():
    pipeline.processed_files.clear()
    print("✓ Cleared in-memory processed files")

print("\n✅ Reset complete. Now run CELL 10 again to reprocess all files.")
print("   This will extract salesperson names correctly for all 496 files.")

In [ ]:
# CELL 10: Run the Pipeline
print("\n" + "=" * 60)
print("🚀 STARTING PIPELINE EXECUTION")
print("=" * 60)

# Run on your ZIP file
results = pipeline.run_from_zip(Path(config.input_zip))

# Generate report
if results:
    report_gen.generate(results)

# Print summary
print("\n" + "=" * 60)
print("📊 EXTRACTION SUMMARY")
print("=" * 60)
print(f"✅ Total processed: {len(results)}")
print(f"⚠️ Needs review: {sum(1 for r in results if r.needs_review)}")
if results:
    avg_conf = sum(r.overall_confidence for r in results) / len(results)
    print(f"📈 Average confidence: {avg_conf:.1%}")

# Show first 10 results
print("\n📋 First 10 extractions:")
print("-" * 100)
print(f"{'Qutation No':<15} {'Client':<30} {'Total Amount':<15} {'Confidence':<10} {'Review':<8}")
print("-" * 100)
for r in results[:10]:
    qno = r.get_value('quotation_no') or 'N/A'
    client = (r.get_value('client') or 'N/A')[:28]
    total = f"₹{r.get_value('total_amount'):,.0f}" if r.get_value('total_amount') else 'N/A'
    conf = f"{r.overall_confidence:.0%}"
    review = "⚠️" if r.needs_review else "✅"
    print(f"{qno:<15} {client:<30} {total:<15} {conf:<10} {review:<8}")
print("-" * 100)

# Show sample of fields extracted
if results:
    print("\n📋 Sample extraction from first file:")
    print("-" * 60)
    first = results[0]
    print(f"Filename: {first.filename}")
    print(f"Quotation No: {first.get_value('quotation_no')}")
    print(f"Client: {first.get_value('client')}")
    print(f"City: {first.get_value('city')}")
    print(f"Product: {first.get_value('product_type')}")
    print(f"Total Amount: ₹{first.get_value('total_amount'):,.0f}" if first.get_value('total_amount') else "Total Amount: N/A")
    print(f"Date: {first.get_value('date')}")
    print(f"Salesperson: {first.get_value('salesperson')}")
    print(f"Confidence: {first.overall_confidence:.0%}")
    print(f"Needs Review: {'Yes' if first.needs_review else 'No'}")
    if first.review_reasons:
        print(f"Review Reasons: {', '.join(first.review_reasons)}")
    print("-" * 60)

# Save results summary
summary_file = OUTPUT_DIR / 'pipeline_summary.json'
summary_data = {
    'total_processed': len(results),
    'needs_review': sum(1 for r in results if r.needs_review),
    'average_confidence': avg_conf if results else 0,
    'version': config.version,
    'timestamp': datetime.now().isoformat(),
    'files': [r.filename for r in results]
}
with open(summary_file, 'w') as f:
    json.dump(summary_data, f, indent=2)

print(f"\n💾 Summary saved to: {summary_file}")
print("\n✅ Pipeline execution complete!")

In [ ]:
# CELL 11: Interactive Review UI (FIXED - handles empty review list)
import ipywidgets as widgets
from IPython.display import display, clear_output

class ReviewUI:
    """Interactive UI for reviewing and correcting extractions"""

    def __init__(self, records: List[QuotationRecord]):
        self.records = [r for r in records if r.needs_review]
        self.current_index = 0
        self.corrections = []

        if not self.records:
            print("✅ No records need review!")
            print(f"📊 Total processed: {len(records)} records, all clean!")
            # Generate report directly
            if records:
                report_gen.generate(records)
            return

        print(f"⚠️ {len(self.records)} records need review")
        self._create_widgets()
        self._update_display()
        display(self.container)

    def _create_widgets(self):
        # Navigation
        self.prev_btn = widgets.Button(description='◀ Previous', button_style='info')
        self.next_btn = widgets.Button(description='Next ▶', button_style='info')
        self.prev_btn.on_click(self._prev)
        self.next_btn.on_click(self._next)

        # Progress
        self.progress = widgets.HTML()

        # File info (NEW - shows which file)
        self.file_label = widgets.HTML()

        # All fields with clear labels
        self.fields = {}
        fields_config = [
            ('quotation_no', '📄 Quotation No:', 30),
            ('client', '🏢 Client:', 50),
            ('city', '📍 City:', 30),
            ('product_type', '🔧 Product:', 20),
            ('total_amount', '💰 Total Amount (₹):', 20),
            ('subtotal', '📊 Subtotal (₹):', 20),
            ('gst', '🧾 GST (₹):', 20),
            ('date', '📅 Date:', 20),
            ('salesperson', '👤 Salesperson:', 30),
            ('area_sqft', '📐 Area (sq ft):', 15),
            ('rate', '💵 Rate (₹/sq ft):', 15),
            ('discount', '🎯 Discount (₹):', 20),
        ]

        for field, label, width in fields_config:
            self.fields[field] = widgets.Text(
                description=label,
                layout=widgets.Layout(width=f'{width}%'),
                style={'description_width': '120px'}
            )

        # Confidence and review reasons
        self.confidence_label = widgets.HTML()
        self.reasons_output = widgets.Output()

        # Actions
        self.approve_btn = widgets.Button(description='✓ Approve', button_style='success', layout=widgets.Layout(width='150px'))
        self.correct_btn = widgets.Button(description='✎ Correct & Save', button_style='primary', layout=widgets.Layout(width='150px'))
        self.skip_btn = widgets.Button(description='⏭ Skip', button_style='warning', layout=widgets.Layout(width='150px'))

        self.approve_btn.on_click(self._approve)
        self.correct_btn.on_click(self._correct)
        self.skip_btn.on_click(self._skip)

        # Layout
        self.container = widgets.VBox([
            widgets.HTML("<h3>📝 Quotation Review Interface</h3>"),
            widgets.HBox([self.prev_btn, self.progress, self.next_btn]),
            widgets.HTML("<hr>"),
            widgets.HTML("<b>📁 File Information:</b>"),
            self.file_label,
            widgets.HTML("<br><b>📊 Extracted Data:</b>"),
            widgets.VBox(list(self.fields.values())),
            widgets.HTML("<br><b>📈 Quality Metrics:</b>"),
            self.confidence_label,
            self.reasons_output,
            widgets.HTML("<hr>"),
            widgets.HBox([self.approve_btn, self.correct_btn, self.skip_btn])
        ])

    def _update_display(self):
        if self.current_index >= len(self.records):
            clear_output(wait=True)
            print("✅ All reviews complete!")
            print(f"📊 Summary: {len(self.records)} records reviewed")

            # Generate final report
            if hasattr(self, 'all_records') and self.all_records:
                report_gen.generate(self.all_records)
            return

        record = self.records[self.current_index]

        # Update progress
        self.progress.value = f"<b>Record {self.current_index + 1} of {len(self.records)}</b>"

        # Show filename prominently
        self.file_label.value = f"""
        <div style="background-color: #f0f0f0; padding: 10px; border-radius: 5px;">
            <b>📄 Filename:</b> {record.filename}<br>
            <b>🔧 Extraction method:</b> {record.extraction_method}<br>
            <b>⏱ Extracted at:</b> {record.extracted_at}
        </div>
        """

        # Populate fields
        for field_name, widget in self.fields.items():
            value = record.get_value(field_name)
            widget.value = str(value) if value is not None else ''

        # Update confidence
        conf_color = 'green' if record.overall_confidence > 0.7 else 'orange' if record.overall_confidence > 0.5 else 'red'
        self.confidence_label.value = f"""
        <div style="background-color: #e8f4f8; padding: 10px; border-radius: 5px;">
            <b>Overall Confidence:</b> <span style="color: {conf_color}; font-weight: bold;">{record.overall_confidence:.0%}</span><br>
            <b>Individual Field Confidence:</b><br>
            {'<br>'.join([f'&nbsp;&nbsp;- {field}: {getattr(record, field).confidence:.0%}' for field in ['quotation_no', 'client', 'total_amount', 'date'] if getattr(record, field) and getattr(record, field).confidence > 0])}
        </div>
        """

        # Show review reasons
        with self.reasons_output:
            clear_output(wait=True)
            if record.review_reasons:
                print("⚠️ Review Reasons:")
                for reason in record.review_reasons:
                    print(f"   • {reason}")
            else:
                print("✅ No specific issues flagged")

    def _approve(self, b):
        """Approve as-is"""
        self.current_index += 1
        self._update_display()

    def _correct(self, b):
        """Save corrections and move to next"""
        record = self.records[self.current_index]
        corrections_made = []

        # Check each field for changes
        for field_name, widget in self.fields.items():
            original_value = record.get_value(field_name)
            new_value = widget.value.strip()

            if new_value and new_value != str(original_value):
                # Convert to appropriate type if needed
                if field_name in ['total_amount', 'subtotal', 'gst', 'area_sqft', 'rate', 'discount']:
                    try:
                        new_value = float(new_value.replace(',', ''))
                    except:
                        continue

                # Save correction to database
                correction_db.add(field_name, str(original_value), str(new_value))
                corrections_made.append(f"{field_name}: '{original_value}' → '{new_value}'")

                # Update the record
                if hasattr(record, field_name) and getattr(record, field_name):
                    getattr(record, field_name).value = new_value
                    getattr(record, field_name).confidence = 0.95

        if corrections_made:
            print(f"✅ Saved corrections for {record.filename}:")
            for corr in corrections_made:
                print(f"   • {corr}")
        else:
            print(f"ℹ️ No changes made to {record.filename}")

        # Recalculate confidence
        record.calculate_confidence()

        self.current_index += 1
        self._update_display()

    def _skip(self, b):
        """Skip this record"""
        print(f"⏭️ Skipped: {self.records[self.current_index].filename}")
        self.current_index += 1
        self._update_display()

    def _prev(self, b):
        if self.current_index > 0:
            self.current_index -= 1
            self._update_display()

    def _next(self, b):
        if self.current_index < len(self.records) - 1:
            self.current_index += 1
            self._update_display()

# Launch review UI if there are records needing review
review_records = [r for r in results if r.needs_review]
if review_records:
    ui = ReviewUI(results)
else:
    print("\n✅ All records are clean! No review needed.")
    print("📊 Generating final report...")
    # Store all records in the UI instance for later use
    ReviewUI.all_records = results
    report_gen.generate(results)

In [ ]:
# Cell to test inbox setup - Add this as a new cell
from pathlib import Path

INBOX_PATH = '/content/drive/MyDrive/YOUR_FOLDER_NAMEn_Inbox'

# Check if folder exists
print(f"Inbox exists: {Path(INBOX_PATH).exists()}")

# List files in inbox
if Path(INBOX_PATH).exists():
    files = list(Path(INBOX_PATH).glob('*'))
    print(f"Files in inbox: {len(files)}")
    for f in files[:5]:
        print(f"  - {f.name}")

# Check processed log
PROCESSED_LOG = '/content/drive/MyDrive/YOUR_FOLDER_NAMEn_Output/processed_files.txt'
if Path(PROCESSED_LOG).exists():
    with open(PROCESSED_LOG) as f:
        processed = set(f.read().splitlines())
    print(f"Already processed: {len(processed)} files")
else:
    print("No processed log yet - first run will create it")

In [ ]:
# CELL 12: Quick Commands Reference
print("""
╔══════════════════════════════════════════════════════════════╗
║                    QUICK COMMANDS REFERENCE                   ║
╠══════════════════════════════════════════════════════════════╣
║                                                               ║
║  # Process a ZIP file                                        ║
║  results = pipeline.run_from_zip(Path('/path/to/file.zip'))  ║
║                                                               ║
║  # Process a folder                                          ║
║  files = list(Path('/folder').glob('*.pdf'))                 ║
║  results = pipeline.process_batch(files)                     ║
║                                                               ║
║  # Generate report                                           ║
║  report_gen.generate(results)                                ║
║                                                               ║
║  # Add a manual correction                                   ║
║  correction_db.add('client', 'Nath Awnings', 'Client_121')  ║
║                                                               ║
║  # Export clean data to CSV                                  ║
║  import pandas as pd                                         ║
║  clean_data = []                                             ║
║  for r in results:                                           ║
║      if not r.needs_review:                                  ║
║          clean_data.append({                                 ║
║              'Quotation No': r.get_value('quotation_no'),    ║
║              'Client': r.get_value('client'),                ║
║              'Amount': r.get_value('total_amount')           ║
║          })                                                   ║
║  pd.DataFrame(clean_data).to_csv('clean_quotations.csv')     ║
║                                                               ║
╚══════════════════════════════════════════════════════════════╝
""")

In [ ]:
# CELL 12: Clean Report Generator - Produces Organized Excel Output
import pandas as pd
from datetime import datetime
import re

class CleanReportGenerator:
    """Generates clean, organized Excel reports matching your format"""

    def __init__(self, output_dir: Path):
        self.output_dir = output_dir

    def clean_client_name(self, client: str) -> str:
        """Clean and standardize client names"""
        if not client or client == 'None':
            return ''

        # Remove common garbage
        garbage = ['Quotation', 'SUNFAB', 'SUNZIP', 'SUNLOUVER', 'STARPOD',
                   'SUNLITE', 'PERGOLA', 'Hello Sir', 'Thank you', 'Copy']
        for g in garbage:
            if g.lower() in client.lower():
                # Try to extract meaningful name
                parts = client.split()
                for p in parts:
                    if len(p) > 3 and p not in garbage:
                        return p
                return ''

        # Clean up common issues
        client = re.sub(r'\(1\)$', '', client)
        client = re.sub(r'\.\.\.', '', client)
        client = re.sub(r'\s+', ' ', client).strip()

        # Map known corrections
        corrections = {
            'Desai Enterprises': 'Desai Enterprises',
            'Client_121': 'Client_121',
            'SunTech Solutions': 'SunTech Solutions',
            'Client_029': 'Client_029',
            'Client_030': 'Client_030',
            'Client_017': 'Client_017',
            'Client_083': 'Client_083',
        }

        for key, val in corrections.items():
            if key.lower() in client.lower():
                return val

        return client

    def clean_product(self, product: str) -> str:
        """Standardize product names"""
        if not product:
            return ''

        product_map = {
            # Core products
            'Client_152':                               'Client_009',
            'SUNFAB':                               'Client_009',
            'SUNFAB Fabric Pergola':                'Client_009',
            'SUNfab fabric pergola':                'Client_009',
            'SunFAB- Fabric Pergola':               'Client_009',
            'SunFAB- Fabric Pergola ':              'Client_009',
            'Client_153':                               'Client_004',
            'SUNZIP':                               'Client_004',
            'Motorised SUNZip Screen system':       'Client_004',
            'Motorised Zip Screen system':          'Client_004',
            ' Motorised Zip Screen system':         'Client_004',
            'Motorised Zip Blind':                  'Client_004',
            'Motorised Zipscreen':                  'Client_004',
            'Manual Spring loaded Zip Screen system': 'Client_004',
            'Client_139':                            'Client_001',
            'SUNLOUVER':                            'Client_001',
            'Aluminium Louver Pergola (SUNLOUVER)': 'Client_001',
            'Aluminium Louver Pergola (SUNLOUVER) ': 'Client_001',
            'Aluminium Louver Pergola (SUNLOUVER) with Wood finish powder coating)': 'Client_001',
            'Client_148':                              'Client_067',
            'SUNLITE':                              'Client_067',
            'Client_149':                              'Client_006',
            'STARPOD':                              'Client_006',
            # Fix 7: new product types (Issue 4)
            'Client_094':                       'Client_094',
            'MONSOON BLIND':                        'Client_094',
            'Client_016':                'Client_016',
            'ShearFab Sail Shade':                  'Client_016',
            'SHEARFAB':                             'Client_016',
            'Client_082':                      'Client_082',
            'Sunmax Gazebo':                        'Client_082',
            'SunMax':                               'Client_082',
            'SUNMAX':                               'Client_082',
            'Client_008':              'Client_008',
            'Motorised Zipscreen Skylight Blind':   'Client_008',
            ' Motorised Zipscreen Skylight Blind':  'Client_008',
            'Client_002':       'Client_002',
            'Manual Exterior weather blinds':       'Client_002',
            ' Manual Exterior weather blinds':      'Client_002',
            'WEATHER BLIND':                        'Client_002',
            'EXTERIOR BLIND':                       'Client_002',
        }

        for key, val in product_map.items():
            if key.upper() in str(product).upper():
                return val

        return str(product)

    def format_amount(self, amount) -> str:
        """Format amount with commas"""
        if not amount or amount == 'None':
            return ''
        try:
            return f"{int(float(amount)):,}"
        except:
            return str(amount)

    def extract_salesperson(self, record) -> str:
        """Extract salesperson name cleanly"""
        sp = record.get_value('salesperson')
        if not sp or sp == 'None':
            return ''

        # Clean up common names
        sp = re.sub(r'\s+', ' ', str(sp)).strip()

        known_sps = {
            'Client_095':   'Client_095',
            'Samir S. Desai.':  'Client_095',
            'Samir Desai':      'Client_095',
            'Samir':            'Client_095',
            'Client_096':   'Client_096',
            'Surendra Jedhe ':  'Client_096',
            'Client_103':    'Client_103',
            'Chetna Banot':     'Client_103',
            'Client_135':       'Client_135',
            'Ravi Mehta.':      'Client_135',
            'Client_104':    'Client_104',
            'Rajessh Sharma':   'Client_104',
            'Client_140':        'Client_140',
            'Client_105':    'Client_105',
            'Dharmesh Vora ':   'Client_105',
            'Dharmesh Vora Dharmesh Vora Mob': 'Client_105',
            'Client_128':      'Client_128',
            'Client_119':     'Client_119',
            'Client_120':     'Client_120',
            'Client_106':    'Client_106',
            'Client_107':    'Client_107',
        }

        for key, val in known_sps.items():
            if key.lower() in sp.lower():
                return val

        return sp

    def format_date(self, date_value) -> str:
        """Format date consistently"""
        if not date_value or date_value == 'None':
            return ''

        date_str = str(date_value)

        # Handle formats like "30-July-2025"
        match = re.match(r'(\d{1,2})-([A-Za-z]+)-(\d{4})', date_str)
        if match:
            day, month, year = match.groups()
            month_map = {
                'January': '01', 'February': '02', 'March': '03', 'April': '04',
                'May': '05', 'June': '06', 'July': '07', 'August': '08',
                'September': '09', 'October': '10', 'November': '11', 'December': '12',
                'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 'May': '05', 'Jun': '06',
                'Jul': '07', 'Aug': '08', 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'
            }
            month_num = month_map.get(month, month)
            return f"{day}/{month_num}/{year}"

        return date_str

    def determine_region(self, city: str) -> str:
        """Determine region from city"""
        if not city:
            return ''

        city_lower = city.lower()
        region_map = {
            'mumbai': 'Mumbai',
            'thane': 'Mumbai',
            'navi mumbai': 'Mumbai',
            'andheri': 'Mumbai',
            'worli': 'Mumbai',
            'dadar': 'Mumbai',
            'juhu': 'Mumbai',
            'pali hill': 'Mumbai',
            'lower parel': 'Mumbai',
            'fort': 'Mumbai',
            'sion': 'Mumbai',
            'wadala': 'Mumbai',
            'bengaluru': 'Karnataka',
            'bangalore': 'Karnataka',
            'chennai': 'Tamil Nadu',
            'coimbatore': 'Tamil Nadu',
            'delhi': 'Delhi NCR',
            'new delhi': 'Delhi NCR',
            'gurgaon': 'Delhi NCR',
            'noida': 'Delhi NCR',
            'ahmedabad': 'Gujarat',
            'surat': 'Gujarat',
            'vadodara': 'Gujarat',
            'baroda': 'Gujarat',
            'rajkot': 'Gujarat',
            'pune': 'Maharashtra',
            'nashik': 'Maharashtra',
            'lonavala': 'Maharashtra',
            'kolkata': 'West Bengal',
            'indore': 'Madhya Pradesh',
            'bhopal': 'Madhya Pradesh'
        }

        for key, region in region_map.items():
            if key in city_lower:
                return region

        return city.title()

    def generate_clean_report(self, records: List[QuotationRecord]) -> Path:
        """Generate clean, organized Excel report"""

        clean_data = []

        for r in records:
            # Skip records with no client or no amount (likely invalid)
            if not r.get_value('client') and not r.get_value('total_amount'):
                continue

            row = {
                'Source File': r.filename,
                'Customer Name': self.clean_client_name(r.get_value('client')),
                'Quoted by': self.extract_salesperson(r),
                'Date': self.format_date(r.get_value('date')),
                'Product Type': self.clean_product(r.get_value('product_type')),
                'Quotation Value (Rs.)': self.format_amount(r.get_value('total_amount')),
                'City': r.get_value('city') or '',
                'Region': self.determine_region(r.get_value('city')),
                'Confidence': f"{r.overall_confidence:.0%}" if r.overall_confidence > 0 else ''
            }
            clean_data.append(row)

        # Create DataFrame
        df = pd.DataFrame(clean_data)

        # Fix 3: Deduplicate by leading quotation number, preferring DOCX over PDF
        def get_quot_num(filename):
            m = re.match(r'^(\d+)', str(filename))
            return m.group(1) if m else None

        def ext_priority(filename):
            """Lower = higher priority (DOCX preferred over PDF)."""
            ext = str(filename).lower().rsplit('.', 1)[-1]
            return {'docx': 0, 'doc': 1, 'pdf': 2, 'xlsx': 3, 'xls': 4}.get(ext, 9)

        df['_quot_num'] = df['Source File'].apply(get_quot_num)
        df['_ext_priority'] = df['Source File'].apply(ext_priority)
        # Sort so that within the same quot_num, DOCX rows come first
        df = df.sort_values(['_quot_num', '_ext_priority'])
        # Drop duplicates by quotation number, keeping the first (highest-priority format)
        df = df.drop_duplicates(subset=['_quot_num'], keep='first')
        df = df.drop(columns=['_quot_num', '_ext_priority'])

        # Sort by filename (numerical order)
        def extract_number(filename):
            match = re.search(r'^(\d+)', str(filename))
            return int(match.group(1)) if match else 999999

        df['_sort_key'] = df['Source File'].apply(extract_number)
        df = df.sort_values('_sort_key').drop('_sort_key', axis=1)

        # Filter out rows with no customer name and no amount
        df = df[~(df['Customer Name'].isin(['', 'None']) & df['Quotation Value (Rs.)'].isin(['', 'None']))]

        # Generate report filename
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        report_path = self.output_dir / f'Quotation_Summary_{timestamp}.xlsx'

        # Write to Excel with formatting
        with pd.ExcelWriter(report_path, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='Quotations', index=False)

            # Summary sheet
            summary = pd.DataFrame([
                ['Total Quotations', len(df)],
                ['Total Value (₹)', self.format_amount(df['Quotation Value (Rs.)'].str.replace(',', '').astype(float).sum()) if len(df) > 0 else '0'],
                ['Unique Customers', df['Customer Name'].nunique()],
                ['Products', ', '.join(df['Product Type'].unique()[:5])],
                ['Regions', ', '.join(df['Region'].unique()[:5])],
                ['Generated On', datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
                ['Pipeline Version', config.version]
            ], columns=['Metric', 'Value'])
            summary.to_excel(writer, sheet_name='Summary', index=False)

            # Pivot by region
            if len(df) > 0:
                region_summary = df.groupby('Region').agg({
                    'Quotation Value (Rs.)': lambda x: self.format_amount(x.str.replace(',', '').astype(float).sum()) if len(x) > 0 else '0',
                    'Customer Name': 'count'
                }).rename(columns={'Quotation Value (Rs.)': 'Total Value (₹)', 'Customer Name': 'Count'})
                region_summary.to_excel(writer, sheet_name='By Region')

            # Pivot by product
            if len(df) > 0:
                product_summary = df.groupby('Product Type').agg({
                    'Quotation Value (Rs.)': lambda x: self.format_amount(x.str.replace(',', '').astype(float).sum()) if len(x) > 0 else '0',
                    'Customer Name': 'count'
                }).rename(columns={'Quotation Value (Rs.)': 'Total Value (₹)', 'Customer Name': 'Count'})
                product_summary.to_excel(writer, sheet_name='By Product')

        return report_path

# Create clean report generator
clean_report_gen = CleanReportGenerator(OUTPUT_DIR)
print("✅ Clean report generator ready")

In [ ]:
# CELL 13: Generate Clean, Organized Report (FIXED - handles empty values)
print("\n" + "=" * 60)
print("📊 GENERATING CLEAN REPORT")
print("=" * 60)

def safe_float_convert(value):
    """Safely convert value to float, return 0 if invalid"""
    if value is None or value == '' or value == 'None':
        return 0
    try:
        if isinstance(value, str):
            value = value.replace(',', '')
        return float(value)
    except:
        return 0

# Load results from JSONL if not in memory
if 'results' not in dir() or not results:
    from pathlib import Path
    import json

    results = []
    jsonl_file = OUTPUT_DIR / 'extractions.jsonl'
    if jsonl_file.exists():
        with open(jsonl_file, 'r') as f:
            for line in f:
                if line.strip():
                    data = json.loads(line)
                    record = QuotationRecord(filename=data.get('filename', ''))
                    record.overall_confidence = data.get('overall_confidence', 0)
                    record.needs_review = data.get('needs_review', False)
                    record.extraction_method = data.get('extraction_method', '')
                    record.extracted_at = data.get('extracted_at', '')

                    for field in ['quotation_no', 'date', 'client', 'city', 'product_type',
                                  'total_amount', 'gst', 'subtotal', 'salesperson', 'area_sqft']:
                        if field in data and data[field] and data[field].get('value'):
                            setattr(record, field, ExtractedField(
                                value=data[field]['value'],
                                confidence=data[field].get('confidence', 0),
                                source=data[field].get('source', '')
                            ))
                    results.append(record)

    print(f"📁 Loaded {len(results)} records from JSONL")

# Generate clean report
if results:
    clean_data = []

    for r in results:
        client_val = r.get_value('client')
        amount_val = r.get_value('total_amount')

        if (not client_val or client_val == 'None') and (not amount_val or amount_val == 0):
            continue

        formatted_amount = ''
        if amount_val and amount_val != 'None':
            try:
                amount_float = safe_float_convert(amount_val)
                if amount_float > 0:
                    formatted_amount = f"{amount_float:,.0f}"
            except:
                formatted_amount = str(amount_val) if amount_val else ''

        city = r.get_value('city')
        if city and city == 'None':
            city = ''

        row = {
            'Source File': r.filename,
            'Customer Name': clean_report_gen.clean_client_name(client_val),
            'Quoted by': clean_report_gen.extract_salesperson(r),
            'Date': clean_report_gen.format_date(r.get_value('date')),
            'Product Type': clean_report_gen.clean_product(r.get_value('product_type')),
            'Quotation Value (Rs.)': formatted_amount,
            'City': city if city else '',
            'Region': clean_report_gen.determine_region(city) if city else '',
            'Confidence': f"{r.overall_confidence:.0%}" if r.overall_confidence > 0 else ''
        }
        clean_data.append(row)

    # Create DataFrame
    df = pd.DataFrame(clean_data)

    if len(df) > 0:

        # ── Fix 3: Deduplicate by leading quotation number, preferring DOCX over PDF ──
        def get_quot_num(fn):
            m = re.search(r'^(\d+)', str(fn))
            return m.group(1) if m else str(fn)

        def ext_priority(fn):
            """Lower number = higher priority. DOCX kept over PDF."""
            ext = str(fn).lower().rsplit('.', 1)[-1]
            return {'docx': 0, 'doc': 1, 'pdf': 2, 'xlsx': 3, 'xls': 4}.get(ext, 9)

        df['_quot_num'] = df['Source File'].apply(get_quot_num)
        df['_ext_priority'] = df['Source File'].apply(ext_priority)
        df = df.sort_values(['_quot_num', '_ext_priority'])
        df = df.drop_duplicates(subset=['_quot_num'], keep='first')
        df = df.drop(columns=['_quot_num', '_ext_priority'])
        # ─────────────────────────────────────────────────────────────────────────────

        # Sort by filename (numerical order)
        def extract_number(filename):
            match = re.search(r'^(\d+)', str(filename))
            return int(match.group(1)) if match else 999999

        df['_sort_key'] = df['Source File'].apply(extract_number)
        df = df.sort_values('_sort_key').drop('_sort_key', axis=1)

        # Filter out rows with no customer name and no amount
        df = df[~((df['Customer Name'] == '') & (df['Quotation Value (Rs.)'] == ''))]

        # Generate report filename
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        report_path = OUTPUT_DIR / f'Quotation_Summary_{timestamp}.xlsx'

        # Write to Excel with formatting
        with pd.ExcelWriter(report_path, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='Quotations', index=False)

            # Summary sheet
            total_value = 0
            for val in df['Quotation Value (Rs.)']:
                if val and val != '':
                    try:
                        total_value += float(str(val).replace(',', ''))
                    except:
                        pass

            summary = pd.DataFrame([
                ['Total Quotations', len(df)],
                ['Total Value (₹)', f"{total_value:,.0f}" if total_value > 0 else '0'],
                ['Unique Customers', df['Customer Name'].nunique()],
                ['Products', ', '.join(df['Product Type'].dropna().unique()[:5])],
                ['Regions', ', '.join(df['Region'].dropna().unique()[:5])],
                ['Generated On', datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
                ['Pipeline Version', config.version]
            ], columns=['Metric', 'Value'])
            summary.to_excel(writer, sheet_name='Summary', index=False)

            # Pivot by region (safe)
            try:
                region_data = []
                for region in df['Region'].dropna().unique():
                    region_rows = df[df['Region'] == region]
                    region_total = 0
                    for val in region_rows['Quotation Value (Rs.)']:
                        if val and val != '':
                            try:
                                region_total += float(str(val).replace(',', ''))
                            except:
                                pass
                    region_data.append({
                        'Region': region,
                        'Count': len(region_rows),
                        'Total Value (₹)': f"{region_total:,.0f}" if region_total > 0 else '0'
                    })
                region_df = pd.DataFrame(region_data)
                if len(region_df) > 0:
                    region_df.to_excel(writer, sheet_name='By Region', index=False)
            except Exception as e:
                print(f"⚠️ Could not create region summary: {e}")

            # Pivot by product (safe)
            try:
                product_data = []
                for product in df['Product Type'].dropna().unique():
                    product_rows = df[df['Product Type'] == product]
                    product_total = 0
                    for val in product_rows['Quotation Value (Rs.)']:
                        if val and val != '':
                            try:
                                product_total += float(str(val).replace(',', ''))
                            except:
                                pass
                    product_data.append({
                        'Product Type': product,
                        'Count': len(product_rows),
                        'Total Value (₹)': f"{product_total:,.0f}" if product_total > 0 else '0'
                    })
                product_df = pd.DataFrame(product_data)
                if len(product_df) > 0:
                    product_df.to_excel(writer, sheet_name='By Product', index=False)
            except Exception as e:
                print(f"⚠️ Could not create product summary: {e}")

        print(f"\n✅ Clean report saved to: {report_path}")

        # Preview
        print("\n📋 PREVIEW OF CLEAN REPORT:")
        print("=" * 120)
        print(f"{'File':<50} {'Customer':<30} {'Amount':<15} {'City':<15}")
        print("=" * 120)
        for _, row in df.head(15).iterrows():
            filename = row['Source File'][:45] + '...' if len(str(row['Source File'])) > 45 else str(row['Source File'])
            customer = str(row['Customer Name'])[:28] if row['Customer Name'] else ''
            amount = str(row['Quotation Value (Rs.)']) if row['Quotation Value (Rs.)'] else ''
            city = str(row['City'])[:12] if row['City'] else ''
            print(f"{filename:<50} {customer:<30} {amount:<15} {city:<15}")
        print("=" * 120)

        print(f"\n📊 Total quotations: {len(df)}")
        print(f"💰 Total value: ₹{total_value:,.0f}")
        print(f"🏢 Unique customers: {df['Customer Name'].nunique()}")

        from google.colab import files
        files.download(str(report_path))

    else:
        print("⚠️ No valid records to report")
else:
    print("⚠️ No results found. Run CELL 10 first.")

In [ ]:
# CELL: Intelligent Inbox Processor - No Duplicates, Cross-Checks Both Sources
# Add this cell AFTER Cell 13 (Clean Report Generator)

import json
import hashlib
from pathlib import Path
from datetime import datetime
from typing import List

class IntelligentInboxProcessor:
    """Processes inbox files without duplicates across ZIP and previous inbox runs"""

    def __init__(self, output_dir: Path):
        self.output_dir = output_dir
        self.processed_log = output_dir / 'all_processed_files.txt'  # Single source of truth
        self.results_jsonl = output_dir / 'extractions.jsonl'
        self._load_processed_files()

    def _load_processed_files(self):
        """Load ALL previously processed files from log"""
        self.processed_files = set()
        if self.processed_log.exists():
            with open(self.processed_log, 'r') as f:
                self.processed_files = set(line.strip() for line in f if line.strip())

        # Also check JSONL for any files not in log
        if self.results_jsonl.exists():
            with open(self.results_jsonl, 'r') as f:
                for line in f:
                    if line.strip():
                        try:
                            data = json.loads(line)
                            filename = data.get('filename', '')
                            if filename:
                                self.processed_files.add(filename)
                        except:
                            pass

        print(f"📋 Loaded {len(self.processed_files)} already processed files")

    def _save_processed_file(self, filename):
        """Mark a file as processed"""
        self.processed_files.add(filename)
        with open(self.processed_log, 'a') as f:
            f.write(filename + '\n')

    def get_new_files(self, inbox_path: str) -> List[Path]:
        """Get only files that haven't been processed before"""
        inbox = Path(inbox_path)
        if not inbox.exists():
            print(f"⚠️ Inbox folder not found: {inbox_path}")
            return []

        supported_exts = {'.pdf', '.docx', '.xlsx', '.xls', '.jpg', '.jpeg', '.png'}
        all_files = []
        for ext in supported_exts:
            all_files.extend(inbox.rglob(f'*{ext}'))
            all_files.extend(inbox.rglob(f'*{ext.upper()}'))

        all_files = list(set(all_files))

        # Filter to ONLY new files
        new_files = []
        skipped_duplicates = 0

        for fp in all_files:
            if fp.name in self.processed_files:
                skipped_duplicates += 1
            else:
                # Also check by file hash for safety (handles renamed files)
                if not self._is_duplicate_by_content(fp):
                    new_files.append(fp)
                else:
                    skipped_duplicates += 1

        print(f"📁 Found: {len(all_files)} total, {len(new_files)} new, {skipped_duplicates} already processed")
        return new_files

    def _is_duplicate_by_content(self, filepath: Path, threshold: float = 0.95) -> bool:
        """Check if file content already exists in results (handles renamed files)"""
        try:
            # Get file size and first 1KB as fingerprint
            stat = filepath.stat()
            fingerprint = f"{stat.st_size}_{hashlib.md5(open(filepath, 'rb').read(1024)).hexdigest()}"

            # Check against existing records
            if hasattr(self, '_content_fingerprints'):
                return fingerprint in self._content_fingerprints

            # Build fingerprint cache from existing results
            self._content_fingerprints = set()
            if self.results_jsonl.exists():
                with open(self.results_jsonl, 'r') as f:
                    for line in f:
                        if line.strip():
                            try:
                                data = json.loads(line)
                                if 'file_hash' in data:
                                    self._content_fingerprints.add(data['file_hash'])
                            except:
                                pass
            return fingerprint in self._content_fingerprints
        except:
            return False

    def process_inbox(self, inbox_path: str, pipeline_instance) -> List:
        """Process only new files from inbox"""
        new_files = self.get_new_files(inbox_path)

        if not new_files:
            print("\n✅ No new files to process. Inbox is up to date!")
            return []

        print(f"\n📦 Processing {len(new_files)} new file(s) from inbox...")
        print("=" * 50)

        results = []
        for i, fp in enumerate(new_files, 1):
            print(f"  [{i}/{len(new_files)}] {fp.name[:60]}...", end=" ")

            try:
                # Process the file using existing pipeline
                record = pipeline_instance.process_file(fp)
                if record:
                    results.append(record)
                    self._save_processed_file(fp.name)
                    print(f"✅ Done")
                else:
                    print(f"⚠️ Skipped (not a quotation)")
                    self._save_processed_file(fp.name)
            except Exception as e:
                print(f"❌ Failed: {str(e)[:50]}")

        print("=" * 50)
        print(f"✅ Successfully processed: {len(results)} new files")

        return results


# ============================================================
# RUN THE INTELLIGENT INBOX PROCESSOR
# ============================================================

print("\n" + "=" * 60)
print("📥 INTELLIGENT INBOX PROCESSOR")
print("=" * 60)

# Initialize the processor
inbox_processor = IntelligentInboxProcessor(OUTPUT_DIR)

# Define inbox path
INBOX_PATH = '/content/drive/MyDrive/YOUR_FOLDER_NAMEn_Inbox'

# Process only new files
new_results = inbox_processor.process_inbox(INBOX_PATH, pipeline)

# If new files were processed, regenerate reports
if new_results:
    print("\n📊 Regenerating reports with new data...")

    # Load all existing results
    all_records = []
    jsonl_file = OUTPUT_DIR / 'extractions.jsonl'
    if jsonl_file.exists():
        with open(jsonl_file, 'r') as f:
            for line in f:
                if line.strip():
                    try:
                        data = json.loads(line)
                        # Reconstruct QuotationRecord
                        record = QuotationRecord(filename=data.get('filename', ''))
                        record.overall_confidence = data.get('overall_confidence', 0)
                        record.needs_review = data.get('needs_review', False)
                        record.extraction_method = data.get('extraction_method', '')
                        record.extracted_at = data.get('extracted_at', '')

                        # Map fields
                        for field in ['quotation_no', 'date', 'client', 'city', 'product_type',
                                      'total_amount', 'gst', 'subtotal', 'salesperson', 'area_sqft']:
                            if field in data and data[field] and data[field].get('value'):
                                setattr(record, field, ExtractedField(
                                    value=data[field]['value'],
                                    confidence=data[field].get('confidence', 0),
                                    source=data[field].get('source', '')
                                ))
                        all_records.append(record)
                    except:
                        pass

    # Generate fresh reports
    if all_records:
        report_gen.generate(all_records)
        print(f"✅ Reports updated with {len(all_records)} total records")
else:
    print("\n✅ No action needed. All files are already processed.")

# Show summary
print("\n" + "=" * 60)
print("📊 INBOX STATUS SUMMARY")
print("=" * 60)
print(f"📁 Inbox location: {INBOX_PATH}")
print(f"📋 Total processed files (all time): {len(inbox_processor.processed_files)}")
print(f"🆕 New files waiting: 0 (all caught up)")
print(f"💾 Processed log: {inbox_processor.processed_log}")

In [ ]:
# CELL 15: RESET ALL PROCESSED DATA (Use for Fresh Start)
# ⚠️ WARNING: This will clear all processed files and force reprocessing

import json
import shutil
from pathlib import Path

print("\n" + "=" * 60)
print("⚠️ RESET ALL PROCESSED DATA")
print("=" * 60)
print("This will clear:")
print("  • Processed checkpoint")
print("  • Inbox processed log")
print("  • Results JSONL file")
print("  • In-memory processed files")
print("  • Corrections database (optional)")
print("\n⚠️ WARNING: This action cannot be undone!")
print("=" * 60)

# Ask for confirmation
confirm = input("\nType 'RESET' to confirm: ")

if confirm != 'RESET':
    print("\n❌ Reset cancelled.")
else:
    print("\n🔄 Resetting...")

    # 1. Clear processed checkpoint
    checkpoint_file = OUTPUT_DIR / 'processed_checkpoint.json'
    if checkpoint_file.exists():
        checkpoint_file.unlink()
        print("✓ Cleared processed checkpoint")

    # 2. Clear inbox processed log
    processed_log = OUTPUT_DIR / 'inbox_processed_files.txt'
    if processed_log.exists():
        processed_log.unlink()
        print("✓ Cleared inbox processed log")

    # 3. Clear all processed log
    all_processed = OUTPUT_DIR / 'all_processed_files.txt'
    if all_processed.exists():
        all_processed.unlink()
        print("✓ Cleared all processed files log")

    # 4. Clear results JSONL
    jsonl_file = OUTPUT_DIR / 'extractions.jsonl'
    if jsonl_file.exists():
        jsonl_file.unlink()
        print("✓ Cleared results JSONL")

    # 5. Clear in-memory processed files from pipeline
    if 'pipeline' in dir() and hasattr(pipeline, 'processed_files'):
        pipeline.processed_files.clear()
        print("✓ Cleared in-memory processed files")

    # 6. Optional: Clear corrections database
    clear_corrections = input("\nClear corrections database as well? (y/n): ")
    if clear_corrections.lower() == 'y':
        corrections_db = OUTPUT_DIR / 'corrections.db'
        if corrections_db.exists():
            corrections_db.unlink()
            print("✓ Cleared corrections database")

    # 7. Optional: Clear extracted files folder
    clear_extracted = input("\nClear extracted files folder? (y/n): ")
    if clear_extracted.lower() == 'y':
        extracted_dir = OUTPUT_DIR / 'extracted'
        if extracted_dir.exists():
            shutil.rmtree(extracted_dir)
            print("✓ Cleared extracted files folder")

    print("\n" + "=" * 60)
    print("✅ RESET COMPLETE")
    print("=" * 60)
    print("\nNext steps:")
    print("  1. Run CELL 10 to reprocess ZIP file")
    print("  2. Run CELL 14 to reprocess Inbox folder")
    print("  3. All files will be processed fresh")

In [ ]:
# CELL 15.5: RESET INBOX ONLY (Keep ZIP processing intact)

print("\n" + "=" * 60)
print("🔄 RESET INBOX ONLY")
print("=" * 60)

# Clear only inbox-related files
processed_log = OUTPUT_DIR / 'inbox_processed_files.txt'
if processed_log.exists():
    processed_log.unlink()
    print("✓ Cleared inbox processed log")

all_processed = OUTPUT_DIR / 'all_processed_files.txt'
if all_processed.exists():
    all_processed.unlink()
    print("✓ Cleared all processed log")

print("\n✅ Inbox reset complete!")
print("Next run of CELL 14 will reprocess all inbox files.")